<p class="lead">Este notebook es la edición en R de la segunda unidad del curso. Las diapositivas de clase son un mapa; este texto es el territorio. La Unidad 1 terminó con un clasificador "entrenado" calculando dos medias; esta unidad se pregunta qué significa entrenar en general —qué es, para una máquina, <em>aprender</em>— y responde con una definición (Sección 1), como un paso en la práctica de la ciencia de datos (Sección 2), como una familia de paradigmas (Secciones 3–5) y como un algoritmo que puedes ejecutar a mano (Secciones 6–7). El plan oficial omite la regresión, el problema de aprendizaje más simple y aquel en que el modelo, la pérdida, la optimización y el sobreajuste pueden verse sin distracciones; por eso la regresión es el núcleo de esta unidad, y la regresión logística —un modelo de regresión usado como clasificador— es el primer clasificador que se estudia en forma rigurosa. Cada figura y cada número del texto los produce el código R que puedes ver y modificar.</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Enunciar la definición de aprendizaje de Mitchell, identificar la tarea, el desempeño y la experiencia en un problema clínico, y distinguir la inferencia inductiva de la deductiva.</li>
<li>Definir ciencia de datos, datos estructurados y no estructurados y los siete pasos de un proyecto, y ubicar el aprendizaje automático dentro de ellos.</li>
<li>Distinguir el aprendizaje supervisado, no supervisado, por refuerzo, semisupervisado y autosupervisado con un ejemplo biomédico de cada uno; usar correctamente "clase", "etiqueta" y "variable objetivo" (target).</li>
<li>Definir un clasificador y su frontera de decisión, y relacionar los enfoques estadístico, neuronal y evolutivo con la Unidad 1.</li>
<li>Deducir y resolver las ecuaciones normales, interpretar coeficientes, \(R^2\) y residuos; implementar el descenso de gradiente y explicar la tasa de aprendizaje; reconocer el sobreajuste y explicar ridge.</li>
<li>Explicar la sigmoide, la pérdida logarítmica (log-loss) y el umbral de la regresión logística, calcular una probabilidad a mano y extenderla a varias clases.</li>
<li>Decir qué aprenden k-means y la iteración de valores, y a partir de qué.</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>Léelo una vez de principio a fin, ejecutando cada bloque de código (chunk) y leyendo el párrafo que lo interpreta; luego vuelve a él por secciones. Las cajas verdes son ejercicios que cambian una cosa en el bloque de código anterior; las azules, ideas clave; las naranjas, errores frecuentes; las moradas, vínculos con otras unidades o con BGM324. Solo se usan R base y los paquetes <code>MASS</code>, <code>class</code>, <code>e1071</code>, <code>rpart</code>, <code>rpart.plot</code>, <code>nnet</code>, <code>cluster</code>, <code>mlbench</code> y <code>ggplot2</code>; en un entorno de ejecución R nuevo de Google Colab, ejecuta una vez <code>install.packages(c("e1071","rpart.plot","nnet","cluster","mlbench"))</code>. Los conjuntos de datos vienen incluidos en <code>mlbench</code> o en R base, así que no se descarga nada. El Suplemento S0 cubre las herramientas de R y el Suplemento S1 la metodología de evaluación que se anticipa en la Sección 11.</p></div>



In [ ]:
# Configuración: paquetes usados a lo largo de la unidad y una semilla fija para que cada número citado sea reproducible.
suppressPackageStartupMessages({
  library(MASS); library(class); library(e1071); library(rpart); library(rpart.plot)
  library(nnet); library(cluster); library(mlbench); library(ggplot2)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#8a8a8a"
options(digits = 5)
R.version.string



La salida impresa es la versión de R con la que se compilaron (knit) estos apuntes. R es un lenguaje construido para los datos —la estadística, el modelado y los gráficos forman parte del lenguaje en lugar de estar añadidos desde fuera<sup class="c"><a href="#r36">[36]</a></sup>— y es el lenguaje que usa el plan oficial de laboratorio, de modo que estos notebooks sirven también como material de laboratorio; el texto complementario en R para todo el curso es *An Introduction to Statistical Learning with Applications in R*.<sup class="c"><a href="#r39">[39]</a></sup>

## 1. Qué significa aprender

### 1.1 La definición de Mitchell

La definición operativa de aprendizaje en el campo es la de Tom Mitchell:<sup class="c"><a href="#r1">[1]</a></sup>

<div class="box key"><h4>Definición (Mitchell, 1997)</h4><p>Se dice que un programa computacional <strong>aprende</strong> de la experiencia <em>E</em> con respecto a una clase de tareas <em>T</em> y una medida de desempeño <em>P</em> si su desempeño en las tareas de <em>T</em>, medido por <em>P</em>, mejora con la experiencia <em>E</em>.</p></div>

Es puramente conductual, y convierte la pregunta "¿aprende el sistema?" en tres preguntas concretas que todo artículo de aplicación debería responder.

| Componente | Pregunta | Ejemplo de cáncer de mama (Unidad 1) | Ejemplo de sala hospitalaria |
|---|---|---|---|
| **Tarea T** | ¿Qué debe hacer el sistema? | Asignar <em>maligno</em> o <em>benigno</em> a un aspirado descrito por características nucleares | Predecir si un paciente se deteriorará dentro de 24 h |
| **Desempeño P** | ¿Cómo se mide hacerlo bien? | Concordancia con la biopsia (mejor: las dos tasas de error por separado) | Sensibilidad a una tasa fija de falsas alarmas |
| **Experiencia E** | ¿Qué datos ve? | 683 aspirados etiquetados de Wisconsin | Signos vitales y desenlaces de hospitalizaciones anteriores |

La idea es anterior a la definición. En 1959, el programa de damas de Arthur Samuel mejoraba jugando contra sí mismo y ajustando los pesos de su función de evaluación; él lo llamó "machine learning" (aprendizaje automático): programar computadores para que aprendan de la experiencia, de modo de eliminar gran parte de la programación detallada.<sup class="c"><a href="#r2">[2]</a></sup> El paso de los sistemas expertos a los sistemas que aprenden (Unidad 1) es el paso de escribir la regla a escribir el <em>procedimiento que encuentra la regla</em>.

### 1.2 Los tres ingredientes de un algoritmo de aprendizaje

Todo algoritmo de aprendizaje de este curso se construye a partir de tres partes:

1. **Un modelo (espacio de hipótesis):** una familia de funciones \(f_\theta\) indexada por parámetros \(\theta\) —intercepto y pendiente para una recta, millones de pesos para una red—; el algoritmo solo puede encontrar un miembro de la familia.
2. **Una función de pérdida** \(L(\theta)\): qué tan mal se desempeña \(f_\theta\) en los datos de entrenamiento —errores al cuadrado, clasificaciones erróneas, log-probabilidad negativa de las etiquetas—; hace operativa la \(P\) de Mitchell.
3. **Un optimizador:** la búsqueda de los parámetros que minimizan la pérdida —una fórmula cerrada (Sección 6), un descenso iterativo (Sección 7) o una búsqueda evolutiva (Sección 4, Unidad 7).

Domingos lo resume en aprendizaje = representación + evaluación + optimización; sus otras doce lecciones —lo que cuenta es la generalización, los datos por sí solos no bastan, el sobreajuste tiene muchas caras— esta unidad las ilustra una por una.<sup class="c"><a href="#r3">[3]</a></sup>

<div class="box warn"><h4>Error frecuente</h4><p>"Entrenar" no es "programar la respuesta". El ser humano aporta la forma (modelo), el criterio (pérdida), los datos y las etiquetas; el algoritmo encuentra los parámetros. Cuando un modelo "fue entrenado con 100,000 radiografías", nadie escribió la regla que lee la radiografía, y puede que nadie sea capaz de leerla.</p></div>

### 1.3 Inferencia inductiva y deductiva

El aprendizaje en el sentido de Mitchell es <strong>inferencia inductiva</strong>: de observaciones particulares a una regla general. La <strong>inferencia deductiva</strong> va en sentido contrario, y ambas difieren en todo lo que importa:

| | Deductiva | Inductiva |
|---|---|---|
| **Dirección** | general → específico: de reglas a casos | específico → general: de casos a una regla |
| **Papel** | <em>aplicar</em> el conocimiento que alguien ya tiene | <em>descubrir</em> conocimiento a partir de los datos |
| **Garantía** | conclusión cierta si las premisas son verdaderas | conclusión probable, nunca cierta |
| **Riesgo** | rigidez: falla en casos que las reglas no anticiparon | sobregeneralización: una regla que se ajusta a la muestra, no al mundo |
| **En IA** | sistemas expertos, lógica (Unidad 1, Sección 3.1) | aprendizaje automático (este curso) |

"Todos los mamíferos tienen pulmones; una ballena es un mamífero; por lo tanto, una ballena tiene pulmones" es deducción: cierta, contenida en las premisas. "Todos los mamíferos examinados hasta ahora tienen pulmones; por lo tanto, todos los mamíferos tienen pulmones" es inducción: la forma en que se obtuvo la regla, falible en principio y confiable en la práctica. Las ratas aprenden inductivamente: una que enferma tras probar un cebo nuevo evita ese cebo —y ese sabor— desde entonces, y por eso los cebos envenenados fracasan tan rápido contra las ratas. Un filtro de spam muestra ambas en software. Un filtro <em>deductivo</em> almacena cada mensaje marcado como spam y bloquea las copias exactas: nunca acusa en falso y lo derrota un solo carácter cambiado. Un filtro <em>inductivo</em> aprende qué palabras caracterizan a los mensajes marcados y puntúa los nuevos según ellas: <em>generaliza</em> a mensajes nunca vistos, y a veces se equivoca. Memorizar es deducir a partir de una tabla; aprender es inducir una regla, y la metodología de este curso existe porque la inducción es falible y su confiabilidad debe medirse.

## 2. De la ciencia de datos al aprendizaje automático

### 2.1 Qué es la ciencia de datos

La <strong>ciencia de datos</strong> extrae conocimiento y conclusiones accionables a partir de datos combinando estadística, computación y conocimiento del dominio; sus raíces están en la estadística, y el recuento histórico de Donoho se pregunta qué hay de genuinamente nuevo en ella.<sup class="c"><a href="#r4">[4]</a></sup> El aprendizaje automático es la parte de la ciencia de datos que construye modelos que aprenden de los datos; la ciencia de datos incluye además todo lo anterior (plantear la pregunta; recolectar, limpiar y explorar los datos) y lo posterior (interpretar y comunicar). Sigue el <strong>método científico</strong> —observar, preguntar, recolectar datos diseñados para responder, analizar, concluir, comunicar para que otros puedan verificar— a una escala que la experimentación clásica no enfrentaba. En un proyecto real, el ajuste del modelo ocupa una pequeña fracción del tiempo; el resto se va en los datos.

### 2.2 Tipos de datos y su procedencia

Los <strong>datos estructurados</strong> se ajustan a un modelo de datos predefinido —una tabla con columnas y tipos fijos, como la matriz de datos de la Unidad 1— y residen en bases de datos relacionales consultadas con <strong>SQL</strong> y, a nivel institucional, en un <strong>almacén de datos</strong> (data warehouse), un repositorio curado construido para análisis conocidos. Los <strong>datos no estructurados</strong> no tienen un modelo predefinido —notas clínicas, imágenes, audio, secuencias— y residen como archivos o en bases de datos <strong>NoSQL</strong> y, a gran escala, en un <strong>lago de datos</strong> (data lake) de datos en bruto que solo se interpretan al leerlos. La mayoría de los datos hospitalarios son no estructurados, y el pipeline de reconocimiento de patrones de la Unidad 1 es la maquinaria que los convierte en la tabla estructurada que requiere todo método de este curso. Los datos <strong>cuantitativos</strong> admiten aritmética; los <strong>cualitativos</strong> son categorías, texto o juicios. Los métodos de análisis se nombran según su producto: <strong>clasificación</strong> (una categoría), <strong>regresión</strong> (un número), <strong>agrupamiento</strong> (grupos sin etiquetas) y <strong>minería de datos</strong>, el descubrimiento de patrones como las reglas de asociación (Sección 3.3).

Los datos provienen de <strong>sistemas transaccionales</strong> (registros hospitalarios, de farmacia, de laboratorio), <strong>fuentes públicas</strong> (estadísticas, repositorios como UCI), <strong>experimentos</strong> y ensayos, <strong>bases de datos</strong> curadas (ChEMBL, DrugBank, UniProt), la <strong>literatura</strong> y las <strong>patentes</strong>, <strong>sensores</strong> (monitores, dispositivos portátiles, imagenología) y <strong>medios</strong>. Cada fuente tiene sus propios sesgos.

### 2.3 Propósitos

Los proyectos <strong>describen</strong> un fenómeno (cuántos pacientes fueron rehospitalizados), lo <strong>explican</strong> (qué factores se asocian), lo <strong>predicen</strong> (qué pacientes serán rehospitalizados) y, como consecuencia, <strong>reducen costos</strong>, <strong>mitigan riesgos</strong> y <strong>mejoran la eficiencia</strong>. El aprendizaje automático sirve principalmente a la predicción.

### 2.4 Los siete pasos de un proyecto de ciencia de datos

1. **Identificación del problema.** La pregunta, la unidad de análisis, la variable objetivo, la medida de éxito, la decisión que se informa: la \(T\), la \(P\) y la \(E\) de Mitchell.
2. **Recolección de datos.** Desde las fuentes anteriores, con una base legal y una procedencia documentada.
3. **Limpieza de datos.** Cuantificar los <em>datos faltantes</em> y decidir (descartar, imputar, modelar); detectar los <em>valores atípicos</em> (outliers) y evaluarlos; eliminar registros y columnas no deseados; unificar estructura y formato.
4. **Análisis exploratorio de datos (EDA).** Estadística descriptiva, <em>univariada</em> y <em>multivariada</em> (correlaciones, tablas de contingencia); pruebas de hipótesis cuando se justifiquen; sobre todo, representación gráfica.
5. **Ingeniería de características.** Selección de atributos; <em>balance de clases</em>; agrupación de categorías; <em>normalización</em> (la Sección 7 muestra por qué); <em>correlaciones entre características</em>, ya que las características correlacionadas desestabilizan los modelos lineales (Sección 6.5).
6. **Predicción con aprendizaje automático.** Elegir, ajustar, evaluar honestamente (Sección 11, Suplemento S1), iterar volviendo a los pasos 3–5.
7. **Narración con datos (data storytelling).** Comunicar cifras, incertidumbre y limitaciones a las personas que actuarán, en su propio lenguaje.

Tres bloques de código ilustran los pasos 3–5 con los datos de cáncer de mama de Wisconsin,<sup class="c"><a href="#r5">[5]</a></sup> incluidos en el paquete <code>mlbench</code><sup class="c"><a href="#r37">[37]</a></sup> y cargados aquí para todo el notebook. La versión incluida tiene 683 aspirados completos descritos por nueve características de forma nuclear puntuadas de 1 a 10 y etiquetados como benignos o malignos; como esos datos están limpios, el primer bloque de código borra al azar el 5 % de los valores para ilustrar los datos faltantes.



In [ ]:
# Paso 3, limpieza de datos: curar los datos, introducir valores faltantes a modo de ilustración, marcar valores atípicos por puntaje z.
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                  # eliminar la columna Id
bc <- bc[complete.cases(bc), ]                            # eliminar los 16 registros incompletos
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))  # ordinal -> numérico
y <- ifelse(bc$Class == "benign", 1L, 0L)                 # y: 1 = benigno, 0 = maligno (como en la Unidad 1)
X <- bc[, 1:9]

set.seed(0); miss <- matrix(runif(nrow(X) * 9) < 0.05, nrow(X))   # borrar al azar el 5% de las celdas
cat(sprintf("cells missing: %.1f%% | rows with >=1 missing: %.1f%% | rows surviving 'drop any NA row': %d of %d\n",
            100*mean(miss), 100*mean(rowSums(miss) > 0), sum(rowSums(miss) == 0), nrow(X)))
z <- scale(X)                                             # puntajes z de los datos limpios
cat(sprintf("values beyond 4 SD: %d in %d patients; most extreme feature '%s' at z = %.1f\n",
            sum(abs(z) > 4), sum(rowSums(abs(z) > 4) > 0),
            names(which.max(apply(abs(z), 2, max))), max(abs(z))))



Con el 5 % de las celdas faltantes al azar, alrededor del 40 % de las filas tiene algún vacío, y eliminar las filas incompletas conservaría 408 pacientes de 683; por eso la imputación es la práctica habitual y los datos faltantes se cuantifican <em>por columna</em>. El tamizaje por puntaje z marca 14 valores más allá de cuatro desviaciones estándar en 14 pacientes; si son errores o tumores extremos es algo que debe juzgar el patólogo.



In [ ]:
# Paso 4, análisis exploratorio de datos: medias por clase y estructura de correlación de las nueve características.
means <- t(sapply(X, function(z) tapply(z, bc$Class, mean)))
ratio <- sort(means[, "malignant"] / means[, "benign"], decreasing = TRUE)
cat("features whose malignant mean is the largest multiple of the benign mean:\n")
print(round(ratio[1:4], 2))

corr <- cor(X)
dfp <- data.frame(f1 = rep(colnames(X), 9), f2 = rep(colnames(X), each = 9), r = as.vector(corr))
ggplot(dfp, aes(f1, f2, fill = r)) + geom_tile() +
  geom_text(aes(label = sprintf("%.2f", r)), size = 2.6,
            colour = ifelse(abs(dfp$r) > 0.6, "white", "black")) +
  scale_fill_gradient2(low = BLUE, mid = "white", high = ORANGE, midpoint = 0, limits = c(-1, 1)) +
  labs(title = "Pearson correlation between the nine nuclear features", x = NULL, y = NULL, fill = "r") +
  theme_minimal(base_size = 9) + theme(axis.text.x = element_text(angle = 60, hjust = 1))



<p class="caption"><b>Figura 1.</b> Correlación de Pearson entre las nueve características nucleares, dibujada con <code>ggplot2</code>.<sup class="c"><a href="#r40">[40]</a></sup> El tamaño celular y la forma celular se correlacionan en 0.91 —dos medidas del mismo agrandamiento nuclear—, de modo que un modelo que recibe ambas recibe esa información dos veces (la Sección 6.5 muestra la consecuencia). La tabla muestra que los núcleos desnudos, el tamaño celular y los nucléolos normales son los que más difieren entre clases (las medias malignas son unas cinco veces las medias benignas), y por eso las dos características usadas desde la Unidad 1 en adelante son el grosor del grupo celular (clump thickness) y el tamaño celular.</p>



In [ ]:
# Paso 5, ingeniería de características: balance de clases, estandarización y eliminación de características redundantes.
cat(sprintf("class balance: %d malignant (%.1f%%), %d benign (%.1f%%)\n",
            sum(y == 0), 100*mean(y == 0), sum(y == 1), 100*mean(y == 1)))
scaled <- scale(X)
summ <- data.frame(raw_mean = colMeans(X), raw_sd = apply(X, 2, sd),
                   scaled_mean = round(colMeans(scaled), 10), scaled_sd = apply(scaled, 2, sd))
print(round(summ[c("Cl.thickness", "Cell.size", "Bare.nuclei", "Mitoses"), ], 3))

fc <- abs(cor(X)); drop <- character(0)                   # eliminar una de cada par con |r| > 0.9
for (i in 1:8) for (j in (i+1):9)
  if (fc[i, j] > 0.9 && !(colnames(X)[i] %in% drop)) drop <- union(drop, colnames(X)[j])
cat("features dropped as redundant (|r| > 0.9 with a kept feature):", length(drop), "->", drop, "\n")



Las clases son 35 % malignas y 65 % benignas: un desbalance lo bastante leve para no requerir corrección, pero un clasificador que siempre dice "benigno" ya tiene una exactitud del 65 %, la línea base que toda exactitud reportada debe superar. La estandarización da a cada característica media 0 y desviación estándar 1, de modo que características puntuadas en la misma escala de 1 a 10 pero con dispersiones distintas entren en igualdad de condiciones en los métodos basados en distancias y en gradientes. El tamizaje de redundancia elimina una característica, la forma celular, por ser casi una copia del tamaño celular. Nada de esto es "aprendizaje automático", y todo ello afecta el resultado más que la elección entre los clasificadores de la Sección 5.

<div class="box key"><h4>Ciencia de datos frente a aprendizaje automático</h4><p>El aprendizaje automático es el paso 6 de siete. Un proyecto excelente en el paso 6 y descuidado en los pasos 1, 3 o 7 responde la pregunta equivocada, aprende los artefactos de sus datos o nunca se usa. Los métodos de este curso son necesarios, no suficientes.</p></div>

## 3. Los paradigmas del aprendizaje

Los paradigmas difieren en el <em>tipo de retroalimentación</em> contenida en la experiencia \(E\), no en el algoritmo.



In [ ]:
# Esquema de los tres paradigmas clásicos de aprendizaje (figura conceptual).
set.seed(0)
cen <- rbind(c(-1.5, 0), c(1.5, 0.5))
Xb <- rbind(cbind(rnorm(30, cen[1,1], .8), rnorm(30, cen[1,2], .8)),
            cbind(rnorm(30, cen[2,1], .8), rnorm(30, cen[2,2], .8)))
yb <- rep(0:1, each = 30)
op <- par(mfrow = c(1, 3), mar = c(4, 4, 2, 1))
# supervisado
plot(Xb, col = ifelse(yb == 0, BLUE, ORANGE), pch = 19, cex = .7, xaxt = "n", yaxt = "n",
     xlab = "feature 1", ylab = "feature 2", main = "Supervised: inputs + labels")
abline(a = 0.2, b = 8, lty = 2); legend("bottomleft", c("class 0", "class 1", "boundary"),
     col = c(BLUE, ORANGE, "black"), pch = c(19, 19, NA), lty = c(NA, NA, 2), bty = "n", cex = .8)
# no supervisado
plot(Xb, col = GREY, pch = 19, cex = .7, xaxt = "n", yaxt = "n",
     xlab = "feature 1", ylab = "feature 2", main = "Unsupervised: inputs only")
text(0, 2.6, "find structure: groups, directions, outliers", cex = .8, font = 3)
# por refuerzo
plot(0:10, 0:10, type = "n", axes = FALSE, xlab = "", ylab = "", main = "Reinforcement: actions + rewards")
rect(1, 6.2, 9, 8.8, col = adjustcolor(BLUE, .2), border = NA); text(5, 7.5, "Agent (policy)")
rect(1, 1.2, 9, 3.8, col = adjustcolor(ORANGE, .2), border = NA); text(5, 2.5, "Environment\n(patient, game, robot)")
arrows(7.5, 6.1, 7.5, 3.9, length = .12); text(8.3, 5, "action", cex = .8)
arrows(2.5, 3.9, 2.5, 6.1, length = .12); text(1.6, 5, "state,\nreward", cex = .8)
par(op)



<p class="caption"><b>Figura 2.</b> Izquierda: supervisado —cada entrada viene con la salida correcta y el algoritmo ajusta una función entre ambas—. Centro: no supervisado —las mismas entradas sin etiquetas; se ven dos grupos, y k-means (Sección 10) los encontrará—. Derecha: refuerzo —no hay conjunto de datos al inicio; un agente actúa, observa el nuevo estado y una recompensa escalar, y descubre por ensayo y error qué acciones conducen a una recompensa acumulada alta—.</p>

### 3.1 Aprendizaje supervisado

La experiencia son pares \((\mathbf{x}_i, y_i)\): entrada y salida correcta, aportados por un "supervisor" —un patólogo, un laboratorio, una visita de seguimiento—. La tarea es una función con \(f(\mathbf{x}) \approx y\) en entradas no vistas. Si \(y\) es categórica, se trata de <strong>clasificación</strong>; si \(y\) es numérica (progresión, glucosa, tiempo de supervivencia), de <strong>regresión</strong>. Ambas comparten todo salvo la pérdida y la salida, de modo que las Secciones 6–8 hacen regresión y la Sección 9 convierte esa maquinaria en un clasificador. Las etiquetas codifican exactamente lo que queremos predecir, y por eso la mayor parte del impacto práctico es supervisado; su costo —el tiempo de un radiólogo, una biopsia, un año de seguimiento— es el límite del paradigma.

<div class="box warn"><h4>Terminología: clase, etiqueta, variable objetivo</h4><p>En clasificación, <em>clase</em> y <em>etiqueta</em> son sinónimos de la categoría que se predice; <em>variable objetivo</em> (target) abarca ambas. En regresión, la variable objetivo es un valor continuo y <strong>no</strong> debe llamarse clase: no hay clases en un problema de regresión, y el uso de "clase" en las diapositivas para la salida de la regresión es un desliz. Di <em>variable objetivo</em> o <em>respuesta</em> en regresión, y <em>clase</em> o <em>etiqueta</em> en clasificación.</p></div>

### 3.2 Dos ejemplos de química

Dos problemas del descubrimiento computacional de fármacos —el campo de investigación del laboratorio que preparó este material y el tema de la unidad de QSAR de BGM324— muestran ambas formas. Las instancias son moléculas y las características son descriptores calculados a partir de la estructura: peso molecular (MW), donores (HBD) y aceptores (HBA) de enlaces de hidrógeno, y área de superficie polar (PSA), los descriptores de la "regla de los cinco" de Lipinski para la absorción oral (MW < 500, HBD ≤ 5, HBA ≤ 10; PSA < 140 Å² en refinamientos posteriores).<sup class="c"><a href="#r6">[6]</a></sup>

<strong>Clasificación: ¿es la molécula un fármaco aprobado?</strong> Una etiqueta binaria —aprobado o no por la FDA— construida a partir de fda.gov, DrugBank<sup class="c"><a href="#r7">[7]</a></sup> y el Drug Repurposing Hub (clue.io); dados los descriptores de una molécula nueva, se predice esa etiqueta.

<strong>Regresión: ¿qué tan potente es?</strong> La <strong>IC50</strong> de un compuesto frente a una diana es la concentración a la que inhibe la actividad de la diana en un 50 %; cuanto menor, más potente, de modo que el compuesto A con IC50 = 10 µM es mejor inhibidor que el compuesto B con 1000 µM. Los valores medidos para millones de pares compuesto–diana están curados en ChEMBL<sup class="c"><a href="#r8">[8]</a></sup> y BindingDB. Como la IC50 abarca varios órdenes de magnitud, la variable objetivo habitual es <strong>pIC50 = −log<sub>10</sub>(IC50 en mol/L)</strong>: 10 µM equivale a 5, 1 µM a 6, una unidad corresponde a un factor diez en potencia, y los errores se acercan más a una distribución gaussiana.

El bloque de código reproduce ambas tablas con valores <em>ilustrativos</em> (en los rangos correctos, no de compuestos medidos). Cada una sigue el flujo de trabajo de todo método supervisado: una <em>etapa de entrenamiento</em>, en la que se entregan al algoritmo moléculas con etiquetas o pIC50 conocidos y este ajusta los parámetros de un modelo para reproducirlos; y una <em>etapa de predicción</em>, en la que los descriptores de una molécula nueva se entregan al modelo ajustado, que entrega una etiqueta o un pIC50 predichos.



In [ ]:
# Los dos ejemplos de química como data frames (valores ilustrativos, no datos medidos).
drugs <- data.frame(
  MW = c(180.2, 454.6, 285.3, 812.9, 374.4, 1203.3), HBD = c(1, 2, 1, 7, 3, 12),
  HBA = c(3, 6, 4, 13, 5, 19), PSA = c(63.6, 98.1, 46.5, 212.0, 87.3, 345.9),
  FDA_approved = c("yes", "yes", "yes", "no", "yes", "no"),          # la ETIQUETA (clase)
  row.names = c("mol_01","mol_02","mol_03","mol_04","mol_05","mol_06"))
potency <- data.frame(
  MW = c(312.4, 298.3, 401.5, 356.8, 275.3), HBD = c(2, 1, 3, 2, 1),
  HBA = c(4, 3, 6, 5, 3), PSA = c(78.2, 55.1, 110.4, 92.0, 48.7),
  IC50_uM = c(10.0, 1000.0, 0.25, 3.2, 150.0),                       # el OBJETIVO (continuo)
  row.names = c("cpd_A","cpd_B","cpd_C","cpd_D","cpd_E"))
potency$pIC50 <- round(-log10(potency$IC50_uM * 1e-6), 2)
cat("CLASSIFICATION: instances x features + a categorical label\n"); print(drugs)
cat("\nREGRESSION: instances x features + a continuous target (pIC50 = -log10 IC50 in mol/L)\n"); print(potency)



Ambas tablas tienen la forma de los datos de cáncer de mama y de viviendas que se usan más adelante: las filas son instancias y la última columna es lo que el modelo debe aprender —una categoría en la primera y un número en la segunda—. El compuesto A (pIC50 = 5) supera a B (pIC50 = 3) por dos órdenes de magnitud. Los modelos QSAR reales usan cientos de descriptores, la situación (\(p\) comparable a \(n\)) en la que la regularización de la Sección 8 se vuelve inevitable.

### 3.3 Aprendizaje no supervisado

La experiencia son solo las entradas, \(\{\mathbf{x}_i\}\). Sin una respuesta correcta que aproximar, la tarea se reformula como encontrar estructura, y las diapositivas nombran cuatro subtipos, cada uno parte de la Unidad 5. El <strong>agrupamiento</strong> (clustering) reúne instancias por similitud (Figura 2, centro; Sección 10.1). La <strong>reducción de dimensionalidad</strong> reemplaza las \(p\) características por unas pocas nuevas que conservan la mayor parte de la variación: componentes principales, o las representaciones bidimensionales usadas para datos de célula única. La <strong>selección de características</strong> conserva un subconjunto de las características originales; su motivación es la <strong>maldición de la dimensionalidad</strong>: con un tamaño muestral fijo, agregar características primero ayuda a un clasificador y luego lo perjudica, porque los datos se vuelven dispersos en un espacio de volumen creciente y el modelo ajusta ruido en características irrelevantes; Hughes mostró en 1968 que la exactitud alcanza un máximo en un número óptimo de características y luego disminuye.<sup class="c"><a href="#r9">[9]</a></sup> Cuántas características ayudan se responde con datos reservados (Sección 11); el tamizaje de la Sección 2.4 es su forma más rudimentaria. El <strong>análisis de asociación</strong> busca en datos transaccionales —compras, prescripciones, diagnósticos concurrentes— reglas SI–ENTONCES ("a los pacientes a quienes se prescribe A y B también se les prescribe C") con soporte y confianza suficientes; el algoritmo Apriori es el método clásico,<sup class="c"><a href="#r10">[10]</a></sup> y sus reglas son la contraparte, en minería de datos, de las reglas de los sistemas expertos: descubiertas en lugar de escritas. Sin etiquetas, la medida de desempeño es ambigua —¿qué tan bueno es un agrupamiento?—; esa es la dificultad central del paradigma, y también lo que lo hace barato.

### 3.4 Aprendizaje por refuerzo

La experiencia la generan las acciones del propio aprendiz: un <em>agente</em> observa un <em>estado</em>, elige una <em>acción</em>, recibe una <em>recompensa</em> escalar y un nuevo estado; la tarea es una <em>política</em> —de estados a acciones— que maximice la recompensa acumulada. La retroalimentación es <em>evaluativa</em> (qué tan buena fue la acción, no cuál era la correcta) y <em>diferida</em>; de ahí el problema de <em>asignación de crédito</em> (credit assignment), que consiste en atribuir los resultados a acciones anteriores. El libro de Sutton y Barto es la referencia.<sup class="c"><a href="#r11">[11]</a></sup> AlphaGo fue aprendizaje por refuerzo; en medicina se ha usado, de forma controvertida, para proponer la dosificación de fluidos y vasopresores en la sepsis.<sup class="c"><a href="#r12">[12]</a></sup> La Sección 10.2 implementa los dos algoritmos básicos.

### 3.5 Incorporaciones modernas: aprendizaje semisupervisado y autosupervisado

El <strong>aprendizaje semisupervisado</strong> usa un conjunto etiquetado pequeño junto con uno no etiquetado grande: los datos no etiquetados no pueden nombrar las clases, pero una frontera que pasa por una región poco poblada es más plausible que una que corta un grupo denso.<sup class="c"><a href="#r13">[13]</a></sup> El <strong>aprendizaje autosupervisado</strong> fabrica etiquetas a partir de los propios datos —oculta parte de una entrada (la palabra siguiente, un fragmento enmascarado de una imagen) y la predice— y luego <em>ajusta finamente</em> (fine-tuning) la representación aprendida con un conjunto etiquetado pequeño; así se preentrenan los modelos de lenguaje, y cada vez más así se construyen los modelos de imágenes médicas cuando las etiquetas escasean.<sup class="c"><a href="#r14">[14]</a></sup>

<div class="box key"><h4>Idea clave</h4><p>Los paradigmas se distinguen por la <em>retroalimentación</em> contenida en \(E\), no por el algoritmo. La misma red puede entrenarse de forma supervisada, autosupervisada o por refuerzo. "Usa aprendizaje profundo" nombra la familia del modelo; todavía no dice a partir de qué aprendió el modelo.</p></div>

<div class="box bridge"><h4>Puente hacia la bioinformática</h4><p>Un modelo QSAR es regresión supervisada; el agrupamiento jerárquico de una matriz de expresión es no supervisado; los modelos de lenguaje de proteínas como ESM son autosupervisados, entrenados para predecir residuos enmascarados a partir del contexto.</p></div>

## 4. Enfoques del reconocimiento de patrones

La Unidad 1 (Sección 3) distinguió los enfoques simbólico, conexionista, estadístico y evolutivo de la IA. El reconocimiento de patrones hereda los tres últimos; la revisión clásica del campo nombra los enfoques <strong>estadístico</strong> y <strong>neuronal</strong>, y el plan del curso agrega el <strong>evolutivo</strong>.<sup class="c"><a href="#r15">[15]</a></sup>

| Enfoque | Cómo se obtiene la regla de decisión | Métodos | Unidades |
|---|---|---|---|
| **Estadístico** | Modelar las clases, o la probabilidad a posteriori de la clase dadas las características, y estimar los parámetros a partir de la muestra | Clasificadores bayesianos, análisis discriminante, regresión logística, k-NN, árboles, SVM | 2–5 |
| **Neuronal** | Conectar unidades simples con pesos ajustables; ajustarlos iterativamente para reducir el error de entrenamiento<sup class="c"><a href="#r16">[16]</a></sup> | Perceptrón, redes multicapa y convolucionales | 6, 8 |
| **Evolutivo** | Mantener una población de reglas candidatas; puntuar, seleccionar, recombinar, mutar, repetir<sup class="c"><a href="#r17">[17]</a></sup> | Algoritmos genéticos, estrategias evolutivas | 7 |

Aunque históricamente provienen de disciplinas distintas, en la práctica son tres maneras de buscar lo mismo. El bloque de código separa los aspirados malignos de los benignos con las dos características grosor del grupo celular y tamaño celular, usando un representante de cada enfoque: el análisis discriminante lineal calcula una recta en forma cerrada a partir de las medias de clase y una covarianza combinada;<sup class="c"><a href="#r38">[38]</a></sup> el perceptrón parte de una recta aleatoria y la desplaza un poco tras cada caso mal clasificado; una estrategia evolutiva mantiene una población de 20 rectas aleatorias, las puntúa según su exactitud y cruza las cinco mejores con mutación durante 40 generaciones. R no tiene una función de perceptrón ni de estrategia evolutiva, así que ambas se escriben desde cero en R base, tal como la edición en Python escribió las suyas.



In [ ]:
# Un problema, tres enfoques: estadístico (LDA), neuronal (perceptrón), evolutivo (ES).
X2 <- as.matrix(bc[, c("Cl.thickness", "Cell.size")]); colnames(X2) <- c("thickness", "size")
Xs <- scale(X2); yv <- y                                     # benigno = 1, maligno = 0

lda.fit <- lda(Xs, factor(yv))                               # estadístico: forma cerrada
lda.acc <- mean(predict(lda.fit)$class == yv)

tgt <- ifelse(yv == 1, 1, -1); w <- c(0, 0); b <- 0; eta <- 0.1   # neuronal: perceptrón, guiado por el error
for (ep in 1:50) { errs <- 0
  for (i in sample(nrow(Xs))) { s <- sum(w * Xs[i, ]) + b
    if (tgt[i] * s <= 0) { w <- w + eta * tgt[i] * Xs[i, ]; b <- b + eta * tgt[i]; errs <- errs + 1 } }
  passes <- ep; if (errs == 0) break }
per.acc <- mean(((Xs %*% w + b) > 0) == (yv == 1))

acc_w <- function(wv) mean(((Xs %*% wv[1:2] + wv[3]) > 0) == (yv == 1))   # evolutivo: ES (5 + 15)
set.seed(0); pop <- matrix(rnorm(20 * 3), 20, 3); history <- numeric(40)
for (g in 1:40) { fit <- apply(pop, 1, acc_w); parents <- pop[order(-fit)[1:5], ]; history[g] <- max(fit)
  children <- parents[sample(5, 15, replace = TRUE), ] + matrix(rnorm(15 * 3, 0, 0.3), 15, 3)
  pop <- rbind(parents, children) }
w_es <- pop[which.max(apply(pop, 1, acc_w)), ]
cat(sprintf("training accuracy  LDA %.3f | perceptron %.3f (after %d passes) | ES %.3f (generation 1: %.3f)\n",
            lda.acc, per.acc, passes, acc_w(w_es), history[1]))

op <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(Xs, col = ifelse(yv == 0, ORANGE, BLUE), pch = 19, cex = .4,
     xlab = "clump thickness (standardised)", ylab = "cell size (standardised)", main = "Three approaches, one boundary")
xx <- seq(-2, 3, length = 50)
lc <- lda.fit$scaling; lm0 <- (apply(predict(lda.fit)$x, 2, function(.) 0))  # dibujar LDA mediante sus coeficientes
abline(-w[1]/w[2]*0 - b/w[2], -w[1]/w[2], lty = 2)                            # recta del perceptrón
abline(-w_es[3]/w_es[2], -w_es[1]/w_es[2], lty = 3, col = GREEN, lwd = 2)     # recta de la ES
legend("topright", c("perceptron (neural)", "ES (evolutionary)"), lty = c(2, 3),
       col = c("black", GREEN), bty = "n", cex = .8)
plot(1:40, history, type = "l", col = GREEN, lwd = 2, xlab = "generation",
     ylab = "best accuracy in population", main = "Evolution strategy: fitness of the best rule")
par(op)



<p class="caption"><b>Figura 3.</b> Izquierda: los 683 aspirados en el espacio de características estandarizado y las fronteras encontradas por un método neuronal (discontinua) y uno evolutivo (punteada verde). Ambas son rectas que atraviesan la zona de solapamiento, con exactitudes de entrenamiento entre 0.93 y 0.95. Derecha: la mejor exactitud de la población evolutiva sube rápidamente y luego se estabiliza. LDA necesitó un solo cálculo; el perceptrón ejecutó las 50 pasadas sin llegar a cero errores, porque las clases se solapan y ninguna recta las separa perfectamente; la estrategia evolutiva usó 800 evaluaciones de exactitud.</p>

El <em>espacio de hipótesis</em> es el mismo —rectas—, así que ninguno puede superar a la mejor recta; los enfoques difieren en <em>cómo</em> buscan y en lo que ofrecen además: probabilidades (estadístico), escalabilidad a millones de parámetros (neuronal), optimización sin gradientes (evolutivo; la Sección 7 explica por qué eso importa). La práctica moderna los combina libremente.

<div class="box key"><h4>Ejercicio 1</h4><p>Cambia la escala de mutación de la estrategia evolutiva de 0.3 a 0.03 y a 3.0. Las mutaciones pequeñas mejoran lentamente; las grandes nunca se estabilizan. Este es el compromiso de la tasa de aprendizaje de la Sección 7 y la tasa de mutación de la Unidad 7.</p></div>

## 5. Qué es un clasificador

### 5.1 Definición y frontera de decisión

Un <strong>clasificador</strong> es una función \(\hat{y} = f(\mathbf{x})\) que va de vectores de características a un conjunto finito de etiquetas \(\{1, \dots, K\}\). Como las etiquetas son finitas y el espacio de características es continuo, todo clasificador divide el espacio en \(K\) <strong>regiones de decisión</strong> separadas por <strong>fronteras de decisión</strong>; todo lo que "sabe" es dónde están esas fronteras. La mayoría de los clasificadores útiles entregan un <strong>puntaje</strong> por clase —idealmente una <strong>probabilidad</strong> \(P(y = k \mid \mathbf{x})\)— y una <em>regla de decisión</em> convierte los puntajes en una etiqueta. La separación importa porque la regla de decisión es donde entran los <em>costos</em> de los errores: un cáncer no detectado cuesta mucho más que una biopsia innecesaria, y "maligno si \(P > 0.5\)" casi nunca es la regla correcta (Sección 9.5).

### 5.2 El catálogo del curso

El plan enumera redes bayesianas, árboles de decisión, k-means, redes neuronales y algoritmos genéticos. El bloque de código entrena un representante de cuatro familias con las mismas dos características y dibuja sus regiones; la quinta, k-means, se trata en la Sección 10 por la razón que se da en la caja de abajo.



In [ ]:
# Cuatro familias de clasificadores sobre las mismas dos características: la forma de sus regiones de decisión.
df2 <- data.frame(X2, Class = factor(yv, labels = c("malignant", "benign")))
set.seed(0); n <- nrow(bc); tr <- sample(n, round(0.7 * n))
gx <- seq(0, 11, length = 160); gy <- seq(0, 11, length = 160)
grid <- expand.grid(thickness = gx, size = gy)
models <- list("Gaussian naive Bayes (Unit 3)" = naiveBayes(Class ~ ., df2[tr, ]),
               "k-NN, k = 15 (Unit 3)"        = NULL,
               "Decision tree, depth 3 (Unit 4)" = rpart(Class ~ ., df2[tr, ], control = rpart.control(maxdepth = 3)),
               "Neural net, 8 units (Unit 6)" = nnet(Class ~ ., df2[tr, ], size = 8, maxit = 500, trace = FALSE))
op <- par(mfrow = c(1, 4), mar = c(4, 3, 2.4, 1))
accs <- numeric(4); nm <- names(models)
for (k in 1:4) {
  if (k == 2) { pr <- knn(scale(X2)[tr, ], as.matrix(scale(X2))[-tr, ], df2$Class[tr], k = 15)
    gpred <- knn(scale(X2)[tr, ], scale(grid, center = attr(scale(X2),"scaled:center"), scale = attr(scale(X2),"scaled:scale")), df2$Class[tr], k = 15)
  } else { m <- models[[k]]
    pr <- if (k == 3) predict(m, df2[-tr, ], type = "class") else predict(m, df2[-tr, ], type = "class")
    gpred <- if (k == 3) predict(m, grid, type = "class") else predict(m, grid, type = "class") }
  accs[k] <- mean(pr == df2$Class[-tr])
  image(gx, gy, matrix(as.integer(gpred == "benign"), 160), col = c("#fbe3d3", "#dfe7f1"),
        xlab = "clump thickness", ylab = if (k == 1) "cell size" else "", main = sprintf("%s\nacc %.3f", nm[k], accs[k]), cex.main = .8)
  points(X2[-tr, ], col = ifelse(df2$Class[-tr] == "benign", BLUE, ORANGE), pch = 19, cex = .35)
}
par(op)



<p class="caption"><b>Figura 4.</b> Regiones de decisión (naranja maligno, azul benigno) de cuatro familias entrenadas con los mismos 478 casos y evaluadas en los mismos 205 casos reservados (puntos). Bayes ingenuo produce curvas suaves a partir de gaussianas por clase; k vecinos más cercanos, una frontera irregular que sigue a la mayoría local; el árbol, rectángulos paralelos a los ejes, en que cada decisión compara una característica con un umbral; la red, una curva suave de forma libre. Las exactitudes de prueba están entre 0.93 y 0.94: ninguna familia es claramente mejor, un hallazgo documentado en muchos estudios comparativos (benchmarks).<sup class="c"><a href="#r18">[18]</a></sup> Las diferencias que importan son la interpretabilidad, el costo, los requisitos de datos y la robustez.</p>

<div class="box warn"><h4>Una corrección al plan: k-means no es un clasificador</h4><p>El plan incluye k-means entre las "técnicas de clasificación". <strong>k-means es un algoritmo de agrupamiento</strong>: divide datos no etiquetados en \(k\) grupos minimizando la varianza dentro de cada grupo, y nunca ve ni predice una etiqueta de clase.<sup class="c"><a href="#r19">[19]</a></sup><sup class="c"><a href="#r20">[20]</a></sup> Su resultado <em>parece</em> una clasificación y comparte una letra con <em>k vecinos más cercanos</em>, pero ambos no tienen nada más en común. Afirmación aceptada: k-means es el método no supervisado canónico (Unidad 5, Sección 10.1); k vecinos más cercanos es un clasificador supervisado (Unidad 3, Figura 4). Del mismo modo, las "redes bayesianas" son un formalismo general del cual los <em>clasificadores</em> bayesianos de la Unidad 3 son el caso particular que desarrolla el curso, y los "algoritmos genéticos" son un método de optimización (Sección 4, Unidad 7) que puede construir clasificadores, pero no es uno.</p></div>

<div class="box key"><h4>Qué hace el resto de esta unidad</h4><p>La Figura 4 muestra lo que los clasificadores <em>producen</em>; nada de lo visto hasta ahora muestra cómo se <em>encuentran</em> sus parámetros, que es lo que significa aprender. Las Secciones 6–9 lo responden para los modelos más simples —una recta para un número, una recta para una probabilidad— con todo detalle. Cada unidad posterior es una variación de los mismos tres pasos.</p></div>

## 6. Regresión lineal: el primer problema de aprendizaje

Para las secciones de regresión, la edición en Python usó un conjunto de datos de progresión de la diabetes;<sup class="c"><a href="#r21">[21]</a></sup> la edición en R usa el conjunto de referencia (benchmark) <strong>Boston housing</strong>, incluido en <code>mlbench</code><sup class="c"><a href="#r37">[37]</a></sup> —el conjunto de datos canónico para enseñar regresión en R y el ejemplo recurrente de *Introduction to Statistical Learning*<sup class="c"><a href="#r39">[39]</a></sup>—, porque muestra con claridad todo el recorrido de la regresión (una sola característica ya explica un tercio de la varianza, y el modelo completo mucho más). Registra 506 sectores censales del Boston de la década de 1970, con trece variables por sector —tasa de criminalidad, número medio de habitaciones (<code>rm</code>), proporción de población de nivel socioeconómico bajo (<code>lstat</code>), razón alumnos–profesor, distancia a los centros de empleo y otras— y, como variable objetivo, el valor mediano de las viviendas ocupadas por sus propietarios, en miles de dólares (<code>medv</code>). La mecánica es idéntica a la de cualquier regresión clínica; la Sección 9 vuelve a los datos médicos.



In [ ]:
# El conjunto de datos Boston housing: 506 sectores censales x 13 variables, y el objetivo medv (valor mediano, $1000s).
data(BostonHousing, package = "mlbench")
bh <- BostonHousing; bh$chas <- as.numeric(as.character(bh$chas))
yb <- bh$medv; Xd <- as.matrix(bh[, setdiff(names(bh), "medv")])
cat(sprintf("X: %d x %d | target medv: mean %.1f, sd %.1f, range %d-%d\n",
            nrow(Xd), ncol(Xd), mean(yb), sd(yb), min(yb), max(yb)))
print(round(t(apply(Xd[, c("rm","lstat","crim","tax","ptratio")], 2,
                    function(z) c(mean = mean(z), sd = sd(z), min = min(z), max = max(z)))), 2))



Son trece variables en escalas muy distintas —habitaciones alrededor de 6, impuesto (tax) en los cientos—, lo que importa en la Sección 7. La variable objetivo tiene media 22.5 y desviación estándar 9.2: un predictor que siempre entrega la media se equivoca típicamente en alrededor de 9.2 (en miles de dólares, $1000s), y cualquier modelo debe hacerlo mejor.

### 6.1 El modelo

La <strong>regresión</strong> es aprendizaje supervisado con una variable objetivo numérica. El modelo más simple hace de la variable objetivo una función lineal de las características más ruido:

\[ \hat{y} = f_{\boldsymbol\beta}(\mathbf{x}) = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p = \mathbf{x}^\top \boldsymbol\beta \]

donde la segunda forma absorbe el intercepto anteponiendo una característica constante \(x_0 = 1\), de modo que \(\mathbf{x}\) y \(\boldsymbol\beta\) tienen longitud \(p + 1\). El espacio de hipótesis es el conjunto de hiperplanos, y los parámetros son los \(p + 1\) coeficientes. El nombre proviene de la "regresión hacia la mediocridad" de la estatura de los hijos descrita por Galton en 1886, que quedó asociado al método que él usó: ajustar una recta.<sup class="c"><a href="#r22">[22]</a></sup>

### 6.2 La pérdida: mínimos cuadrados

La medida clásica de ajuste es la <strong>suma de los residuos al cuadrado</strong>, donde \(r_i = y_i - \hat{y}_i\) es la distancia vertical de la observación a la recta:

\[ L(\boldsymbol\beta) = \sum_{i=1}^{n} \left(y_i - \mathbf{x}_i^\top \boldsymbol\beta\right)^2 = \lVert \mathbf{y} - \mathbf{X}\boldsymbol\beta \rVert^2 \]

donde \(\mathbf{X}\) es la matriz de datos de \(n \times (p+1)\) con una primera columna de unos. Dividida por \(n\) es el <em>error cuadrático medio</em> (MSE), con el mismo minimizador; su raíz cuadrada (RMSE) está en las unidades de \(y\). Los cuadrados son diferenciables, de modo que el mínimo tiene forma cerrada; penalizan más unos pocos errores grandes que muchos pequeños; y, con ruido gaussiano, minimizar el error cuadrático equivale a maximizar la verosimilitud de los datos, el principio detrás de toda pérdida de este curso, que se usa en la Sección 9 para la regresión logística. Legendre publicó el método en 1805, Gauss afirmó usarlo desde 1795, y la disputa es una de las mejor documentadas en la historia de la ciencia.<sup class="c"><a href="#r23">[23]</a></sup>

### 6.3 Ejemplo resuelto a mano: cuatro puntos

Cuatro observaciones con una sola característica \(x\) y una variable objetivo \(y\) (valores redondos inventados):

| observación | \(x\) | \(y\) |
|---|---|---|
| 1 | 20 | 100 |
| 2 | 24 | 120 |
| 3 | 28 | 180 |
| 4 | 32 | 200 |

Para una sola característica, la recta de mínimos cuadrados \(\hat{y} = \beta_0 + \beta_1 x\) tiene la solución de los libros de texto:

\[ \beta_1 = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2}, \qquad \beta_0 = \bar{y} - \beta_1 \bar{x}. \]

**Paso 1 — medias.** \(\bar{x} = (20 + 24 + 28 + 32)/4 = 26\); \(\bar{y} = (100 + 120 + 180 + 200)/4 = 150\).

**Paso 2 — desviaciones.**

| \(i\) | \(x_i - \bar{x}\) | \(y_i - \bar{y}\) | producto | \((x_i - \bar{x})^2\) |
|---|---|---|---|---|
| 1 | −6 | −50 | 300 | 36 |
| 2 | −2 | −30 | 60 | 4 |
| 3 | 2 | 30 | 60 | 4 |
| 4 | 6 | 50 | 300 | 36 |
| **suma** | | | **720** | **80** |

**Paso 3 — coeficientes.** \(\beta_1 = 720/80 = 9\) unidades de \(y\) por unidad de \(x\); \(\beta_0 = 150 - 9 \times 26 = -84\).

**Paso 4 — verificación.** Predicciones \((96, 132, 168, 204)\); residuos \((4, -12, 12, -4)\), que suman cero, como ocurre siempre con los residuos de mínimos cuadrados cuando hay intercepto; suma de cuadrados \(320\), MSE \(80\). El bloque de código confirma la aritmética y que ninguna otra pendiente lo hace mejor.



In [ ]:
# El ejemplo de cuatro puntos: la recta calculada a mano, y que las pendientes cercanas tienen una pérdida mayor.
x4 <- c(20, 24, 28, 32); y4 <- c(100, 120, 180, 200)
b1 <- sum((x4 - mean(x4)) * (y4 - mean(y4))) / sum((x4 - mean(x4))^2)
b0 <- mean(y4) - b1 * mean(x4); resid4 <- y4 - (b0 + b1 * x4)
cat(sprintf("slope = %.1f, intercept = %.1f, residuals = %s, sum = %.1f, MSE = %.1f\n",
            b1, b0, paste(resid4, collapse = ", "), sum(resid4), mean(resid4^2)))
mse <- function(a, s) mean((y4 - (a + s * x4))^2)
cat(sprintf("MSE of hand line: %.1f | MSE with slope 8: %.1f | with slope 10: %.1f\n",
            mse(b0, b1), mse(mean(y4) - 8 * mean(x4), 8), mse(mean(y4) - 10 * mean(x4), 10)))
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(x4, y4, pch = 19, col = BLUE, cex = 1.4, xlab = "feature x", ylab = "target y", main = "The fitted line and its residuals")
abline(b0, b1, col = ORANGE, lwd = 2); segments(x4, y4, x4, b0 + b1 * x4, col = GREY)
legend("topleft", sprintf("y = %.0f + %.0f x", b0, b1), lty = 1, col = ORANGE, bty = "n", cex = .9)
slopes <- seq(4, 14, length = 100)
plot(slopes, sapply(slopes, function(s) mse(mean(y4) - s * mean(x4), s)), type = "l", col = BLUE, lwd = 2,
     xlab = "slope (intercept optimal for each slope)", ylab = "mean squared error", main = "The loss is a parabola; minimum at 9")
points(b1, mse(b0, b1), pch = 19, col = ORANGE, cex = 1.5)
par(op)



<p class="caption"><b>Figura 5.</b> Izquierda: las cuatro observaciones, la recta \(\hat{y} = -84 + 9x\) y los residuos (gris), cuyos cuadrados minimiza el método. Derecha: el MSE en función de la pendiente, con el intercepto reoptimizado para cada pendiente: una parábola con un mínimo único en pendiente 9 y MSE 80. Con un único mínimo, cualquier método que descienda lo encuentra; la regresión logística comparte esta propiedad.</p>

### 6.4 Las ecuaciones normales

Al desarrollar la pérdida en forma matricial e igualar el gradiente a cero:

\[ \nabla_{\boldsymbol\beta} L = -2\,\mathbf{X}^\top \mathbf{y} + 2\,\mathbf{X}^\top \mathbf{X}\,\boldsymbol\beta = \mathbf{0} \;\Longrightarrow\; \mathbf{X}^\top \mathbf{X}\, \hat{\boldsymbol\beta} = \mathbf{X}^\top \mathbf{y} \;\Longrightarrow\; \hat{\boldsymbol\beta} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}. \]

Estas son las <strong>ecuaciones normales</strong>: \(p + 1\) ecuaciones lineales con \(p + 1\) incógnitas, con solución única cuando ninguna característica es combinación lineal exacta de las demás y \(n > p\). El nombre es geométrico: el vector de residuos es <em>normal</em> (perpendicular) a cada columna de \(\mathbf{X}\), de modo que los valores ajustados son la proyección de \(\mathbf{y}\) sobre el espacio de las características. En código nunca se calcula la inversa: <code>solve</code> de R resuelve el sistema directamente, y <code>lm</code> usa una descomposición QR.



In [ ]:
# Ecuaciones normales con una característica (rm = número medio de habitaciones), contrastadas con lm().
xr <- bh$rm; X1 <- cbind(1, xr)
beta1 <- solve(t(X1) %*% X1, t(X1) %*% yb)                   # resolver (X'X) beta = X'y
yhat1 <- X1 %*% beta1; rss <- sum((yb - yhat1)^2); tss <- sum((yb - mean(yb))^2)
cat("X'X =\n"); print(round(t(X1) %*% X1, 0))
cat(sprintf("beta (intercept, slope) = (%.3f, %.3f) | RMSE = %.2f | R^2 = %.3f\n",
            beta1[1], beta1[2], sqrt(rss / length(yb)), 1 - rss / tss))
print(round(coef(lm(yb ~ xr)), 3))                           # lm() coincide exactamente
plot(xr, yb, pch = 19, col = adjustcolor(BLUE, .4), cex = .5, xlab = "average rooms per dwelling",
     ylab = "median home value ($1000s)", main = "Least-squares line: value against rooms")
abline(beta1[1], beta1[2], col = ORANGE, lwd = 2)
legend("topleft", sprintf("y = %.0f + %.1f x", beta1[1], beta1[2]), lty = 1, col = ORANGE, bty = "n")



<p class="caption"><b>Figura 6.</b> Valor mediano frente al número medio de habitaciones para 506 sectores censales y la recta de mínimos cuadrados, con pendiente de alrededor de 9.1: cada habitación adicional se asocia, en promedio, con unos $9,100 más de valor mediano. La dispersión es amplia —RMSE de alrededor de 6.6 frente al 9.2 del predictor que no hace nada—, de modo que las habitaciones por sí solas explican una fracción modesta pero real de la variación (\(R^2 = 0.48\)).</p>

### 6.5 Cómo leer el resultado: coeficientes y R²

Un <strong>coeficiente</strong> indica cómo cambia la predicción con su característica, <em>manteniendo todo lo demás constante</em>, algo vacío con una característica y esencial con trece. El <strong>coeficiente de determinación</strong>

\[ R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2} = 1 - \frac{\text{RSS}}{\text{TSS}} \]

compara la suma de cuadrados residual del modelo con la del predictor que no hace nada, \(\hat{y} = \bar{y}\): 1 significa que todos los puntos están sobre la recta, 0 que no es mejor que la media; en los datos de entrenamiento no puede ser negativo cuando se ajusta un intercepto, en datos nuevos sí puede.



In [ ]:
# Ecuaciones normales con las trece características. El código es el mismo; solo la matriz de diseño es más ancha.
X13 <- cbind(1, Xd); beta13 <- solve(t(X13) %*% X13, t(X13) %*% yb); yhat13 <- X13 %*% beta13
r2_13 <- 1 - sum((yb - yhat13)^2) / tss
cat(sprintf("R^2 with 13 features = %.3f (rooms alone: %.3f) | RMSE = %.2f\n",
            r2_13, 1 - rss / tss, sqrt(mean((yb - yhat13)^2))))
coef <- data.frame(per_unit = beta13[-1], feature_sd = apply(Xd, 2, sd))
coef$effect_of_1sd <- coef$per_unit * coef$feature_sd
print(round(coef[order(-abs(coef$effect_of_1sd)), ], 3))



\(R^2\) sube a 0.74 y el RMSE baja a 4.7. El coeficiente de las habitaciones cae de 9.1 a 3.8: parte de lo que las habitaciones "explicaban" solas se atribuye ahora a variables con las que están correlacionadas. Los coeficientes en bruto no son comparables, así que la tercera columna reescala cada uno por desviación estándar de su característica: la proporción de población de nivel socioeconómico bajo (<code>lstat</code>), la distancia (<code>dis</code>) y la concentración de óxido nítrico son las que más contribuyen negativamente, y las habitaciones y el acceso a autopistas radiales (<code>rad</code>) las que más lo hacen positivamente. Esta lectura de los coeficientes es la base de todo modelo de riesgo multivariable en medicina.

<div class="box warn"><h4>Dos advertencias</h4><p>Un coeficiente describe cómo cambia la <em>predicción</em>, no lo que ocurriría si la característica cambiara. Y cuando las características están fuertemente correlacionadas —aquí el impuesto a la propiedad y el acceso a autopistas (<code>tax</code>, <code>rad</code>) se correlacionan en 0.91—, \(\mathbf{X}^\top\mathbf{X}\) es casi singular y los coeficientes se vuelven inestables: grandes, de signos opuestos, sensibles a unos pocos casos, mientras las predicciones siguen siendo buenas. El par <code>tax</code> (−2.1 por desviación estándar) y <code>rad</code> (+2.7) es esta inestabilidad. La regresión ridge (Sección 8.3) se inventó para ella.<sup class="c"><a href="#r24">[24]</a></sup></p></div>

### 6.6 Gráficos de residuos

\(R^2\) oculta <em>dónde</em> falla el modelo; los residuos lo muestran. Frente a los valores ajustados deberían formar una banda sin estructura —una curva indica no linealidad; un embudo, ruido que crece con la predicción—; su histograma debería ser aproximadamente simétrico.



In [ ]:
# Diagnóstico de residuos del modelo de trece características.
resid13 <- yb - yhat13
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(yhat13, resid13, pch = 19, col = adjustcolor(BLUE, .4), cex = .5, xlab = "fitted value",
     ylab = "residual (observed - fitted)", main = "Residuals against fitted values"); abline(h = 0)
hist(resid13, breaks = 30, col = adjustcolor(BLUE, .7), xlab = "residual", main = "Distribution of residuals"); abline(v = 0)
par(op)
skew <- mean((resid13 - mean(resid13))^3) / sd(resid13)^3
cat(sprintf("residual mean = %.2e (zero by construction) | sd = %.2f | skewness = %.2f | largest |residual| = %.1f\n",
            mean(resid13), sd(resid13), skew, max(abs(resid13))))



<p class="caption"><b>Figura 7.</b> Izquierda: residuos frente a los valores ajustados. A la derecha aparece una franja diagonal de residuos positivos grandes, donde el modelo subestima sistemáticamente los sectores más caros: el valor mediano estaba <em>truncado en 50</em> en la encuesta (un efecto techo que un modelo lineal no puede representar). Derecha: los residuos tienen asimetría positiva (asimetría 1.52), desviación estándar 4.7 y residuo máximo 26. La asimetría es la huella de ese techo; el paso siguiente sería transformar la variable objetivo o usar un modelo no lineal (Unidad 4). La edición en Python vio en sus datos el efecto suelo, la imagen especular de este: la misma lección metodológica.</p>

<div class="box key"><h4>Ejercicio 2</h4><p>(a) Verifica en papel que, para una característica con intercepto, las ecuaciones normales son \(n\beta_0 + \beta_1 \sum x_i = \sum y_i\) y \(\beta_0 \sum x_i + \beta_1 \sum x_i^2 = \sum x_i y_i\), y que al resolverlas se obtienen las fórmulas de la Sección 6.3. (b) Elimina las columnas <code>tax</code> y <code>rad</code> en el bloque de código de trece características y vuelve a ajustar. ¿Cuánto cambian \(R^2\) y los coeficientes restantes?</p></div>

## 7. Descenso de gradiente: cómo aprende una máquina

### 7.1 Por qué no basta una forma cerrada

Casi ningún problema de aprendizaje tiene forma cerrada: la regresión logística (Sección 9) no la tiene, la superficie de pérdida de una red neuronal no tiene fórmula para su mínimo, e incluso la regresión lineal con \(p\) en los cientos de miles no puede construir \(\mathbf{X}^\top\mathbf{X}\). Lo que sí tiene todo problema es una pérdida que se puede <em>evaluar</em> y <em>derivar</em>. El <strong>descenso de gradiente</strong> no usa nada más, entrena prácticamente todos los modelos modernos y es el momento en que "la máquina aprende" deja de ser una metáfora.

### 7.2 La idea

El gradiente \(\nabla L(\theta)\) es el vector de derivadas parciales; apunta cuesta arriba, de modo que su negativo apunta cuesta abajo. Parte desde cualquier punto y repite

\[ \theta^{(t+1)} = \theta^{(t)} - \eta\, \nabla L(\theta^{(t)}) \]

donde \(\eta > 0\) es la <strong>tasa de aprendizaje</strong>; se detiene cuando el gradiente es casi cero, cuando la pérdida deja de cambiar o tras un número fijo de pasos. La tasa de aprendizaje tiene tres regímenes: demasiado pequeña, correcta pero lenta; demasiado grande, cada paso se pasa de largo y la pérdida <em>diverge</em>; entre ambos, convergencia rápida. Para una pérdida cuadrática el límite es exacto —\(\eta < 2/\lambda_{\max}\), siendo el denominador el mayor valor propio de la hessiana— y la velocidad la fija la razón entre el valor propio mayor y el menor, el <em>número de condición</em>, que es grande cuando las características tienen escalas distintas. Por eso el escalamiento de características, opcional para las ecuaciones normales, es casi obligatorio para el descenso de gradiente.

### 7.3 El gradiente de la pérdida de mínimos cuadrados

Para el MSE \(L(\boldsymbol\beta) = \frac{1}{n}\lVert \mathbf{y} - \mathbf{X}\boldsymbol\beta \rVert^2\),

\[ \nabla_{\boldsymbol\beta} L = -\frac{2}{n}\, \mathbf{X}^\top (\mathbf{y} - \mathbf{X}\boldsymbol\beta) = -\frac{2}{n}\, \mathbf{X}^\top \mathbf{r}, \]

la correlación de cada característica con los residuos: si una característica es alta donde el modelo subestima, su coeficiente aumenta. El bloque de código implementa la actualización de tres líneas para una característica —<code>lstat</code>, elegida porque su gran dispersión hace que la superficie de pérdida esté mal condicionada y los tres regímenes se vean con nitidez—, <em>centrada</em> (de modo que el intercepto es la media de \(y\)), con tres tasas de aprendizaje.



In [ ]:
# Descenso de gradiente por lotes (batch) para mínimos cuadrados, a mano. Tres tasas de aprendizaje: demasiado pequeña, adecuada, demasiado grande.
xl <- bh$lstat; xc <- xl - mean(xl); Xc <- cbind(1, xc)       # característica centrada
beta_closed <- solve(t(Xc) %*% Xc, t(Xc) %*% yb)              # forma cerrada para comparar
gradient_descent <- function(Xm, yv, lr, steps = 100) {
  beta <- c(0, 0); path <- matrix(0, steps + 1, 2); losses <- numeric(steps); path[1, ] <- beta
  for (t in 1:steps) { r <- yv - Xm %*% beta; losses[t] <- mean(r^2)
    beta <- beta - lr * (-2 / length(yv) * t(Xm) %*% r); path[t + 1, ] <- beta }
  list(path = path, losses = losses) }
lam_max <- 2 * max(eigen(t(Xc) %*% Xc / length(yb))$values)
cat(sprintf("largest Hessian eigenvalue = %.1f -> converges only if lr < 2/%.1f = %.4f\n", lam_max, lam_max, 2 / lam_max))
runs <- lapply(c(0.002, 0.015, 0.021), function(lr) gradient_descent(Xc, yb, lr))
for (k in 1:3) { lr <- c(0.002, 0.015, 0.021)[k]
  cat(sprintf("lr = %-6g after 100 steps: beta = (%.2f, %.2f)  MSE = %.4g\n",
              lr, runs[[k]]$path[101, 1], runs[[k]]$path[101, 2], runs[[k]]$losses[100])) }
cat(sprintf("closed form:            beta = (%.2f, %.2f)  MSE = %.4g\n",
            beta_closed[1], beta_closed[2], mean((yb - Xc %*% beta_closed)^2)))



Aparecen los tres regímenes. Con \(\eta = 0.002\), la pérdida sigue bajando tras 100 pasos y el intercepto (7.4) está lejos de 22.5, mientras que la pendiente —la dirección de mayor curvatura— convergió hace mucho. Con \(\eta = 0.015\), ambos parámetros coinciden con la forma cerrada y el MSE es igual al mínimo, alrededor de 38.5. Con \(\eta = 0.021\), apenas por encima del umbral de 0.0196, la pendiente oscila con amplitud creciente y la pérdida llega al orden de \(10^{12}\).



In [ ]:
# Curvas de convergencia para las tres tasas de aprendizaje, y la trayectoria de dos de ellas sobre la superficie de pérdida.
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
cols <- c(GREY, BLUE, ORANGE); lrs <- c(0.002, 0.015, 0.021)
plot(NA, xlim = c(1, 100), ylim = c(1, 1e6), log = "y", xlab = "step",
     ylab = "mean squared error (log, clipped)", main = "Loss per step for three learning rates")
for (k in 1:3) lines(pmin(pmax(runs[[k]]$losses, 1), 1e6), col = cols[k], lwd = 1.5)
abline(h = mean((yb - Xc %*% beta_closed)^2), lty = 2)
legend("topright", c("lr = 0.002", "lr = 0.015", "lr = 0.021 (diverges)", "closed-form min"),
       col = c(cols, "black"), lty = c(1, 1, 1, 2), bty = "n", cex = .8)
b0g <- seq(0, 45, length = 120); b1g <- seq(-3, 1, length = 120)
L <- outer(b0g, b1g, Vectorize(function(a, s) mean((yb - (a + s * xc))^2)))
contour(b0g, b1g, L, nlevels = 14, col = GREY, xlab = "intercept", ylab = "slope",
        main = "Descent on the loss surface (MSE contours)")
for (k in 1:2) { p <- runs[[k]]$path; lines(p[, 1], p[, 2], col = cols[k], type = "o", pch = 19, cex = .3) }
points(beta_closed[1], beta_closed[2], pch = 8, col = ORANGE, cex = 2)
par(op)



<p class="caption"><b>Figura 8.</b> Izquierda: MSE por paso para las tres tasas (eje logarítmico). La tasa pequeña desciende de forma sostenida, pero no ha alcanzado el mínimo (línea discontinua) tras 100 pasos; la tasa moderada lo alcanza en unas pocas decenas; la tasa grande disminuye brevemente y luego crece sin límite (recortada en \(10^6\)). Derecha: la superficie de pérdida sobre (intercepto, pendiente). La dirección de la pendiente tiene una curvatura unas cincuenta veces mayor que la dirección del intercepto (los dos ejes se dibujan en escalas muy distintas, por lo que los contornos, en realidad alargados, aquí se ven redondos); la trayectoria de la tasa moderada baja por la dirección empinada de la pendiente en unos pocos pasos oscilantes y luego avanza lentamente por la dirección plana del intercepto. Es el número de condición hecho visible; estandarizar la característica igualaría las curvaturas.</p>

<div class="box key"><h4>Esto es aprender</h4><p>Nada en el bloque de código del descenso de gradiente sabe de viviendas ni de rectas; conoce una pérdida y su gradiente, y camina cuesta abajo. Reemplaza el modelo por una red con un millón de pesos y la pérdida por la de la Sección 9, conserva las tres líneas de la actualización, y tendrás el ciclo de entrenamiento de todo sistema de aprendizaje profundo.</p></div>

### 7.4 Descenso de gradiente estocástico

El gradiente anterior usó los \(n\) casos en cada paso: descenso <em>por lotes</em> (batch). Con un millón de imágenes eso es innecesario: el gradiente calculado sobre un <em>minilote</em> (mini-batch) aleatorio es una estimación ruidosa pero insesgada del gradiente completo. El <strong>descenso de gradiente estocástico</strong> (SGD) hace esto con una tasa de aprendizaje decreciente, de modo que el ruido se promedia; la demostración de convergencia es la aproximación estocástica de Robbins y Monro de 1951.<sup class="c"><a href="#r25">[25]</a></sup> El bloque de código implementa desde cero el SGD por minilotes (R no tiene un regresor SGD incorporado) sobre las trece características, con y sin escalamiento.



In [ ]:
# Descenso de gradiente estocástico por minilotes desde cero, con características sin escalar vs estandarizadas.
sgd <- function(Xm, yv, lr0, epochs = 60, bs = 32) {
  Xa <- cbind(1, Xm); beta <- rep(0, ncol(Xa)); nn <- nrow(Xa); it <- 0
  for (e in 1:epochs) { idx <- sample(nn)
    for (s in seq(1, nn, bs)) { bi <- idx[s:min(s + bs - 1, nn)]; it <- it + 1
      r <- yv[bi] - Xa[bi, ] %*% beta; lr <- lr0 / (1 + it * 1e-3)
      beta <- beta - lr * (-2 / length(bi) * t(Xa[bi, ]) %*% r) } }
  1 - sum((yv - Xa %*% beta)^2) / sum((yv - mean(yv))^2) }
r2_ols <- summary(lm(yb ~ Xd))$r.squared
set.seed(0); r2_raw <- sgd(Xd, yb, 1e-5); set.seed(0); r2_scaled <- sgd(scale(Xd), yb, 0.05)
res <- data.frame(method = c("normal equations", "SGD on raw features", "SGD on standardised features"),
  R2_training = c(sprintf("%.3f", r2_ols),
                  if (is.finite(r2_raw)) sprintf("%.3f", r2_raw) else "diverged",
                  sprintf("%.3f", r2_scaled)))
print(res, row.names = FALSE)



Con características estandarizadas, el SGD alcanza el \(R^2\) de las ecuaciones normales (0.74 frente a 0.74) en unas pocas decenas de pasadas. Con las características en bruto —el impuesto (tax) en las centenas, la criminalidad cerca de cero— diverge: ninguna tasa única sirve para todas las direcciones de una superficie tan alargada. Estandariza antes de usar cualquier método basado en gradientes.

<div class="box key"><h4>Ejercicio 3</h4><p>Reemplaza la característica centrada <code>xc</code> por la estandarizada <code>(xl - mean(xl)) / sd(xl)</code> y vuelve a ejecutar ambos bloques de código: el umbral pasa a ser <code>lr &lt; 1</code> y los contornos, círculos. Encuentra una tasa que alcance el mínimo en menos de diez pasos. Luego ejecuta el descenso por lotes con <code>steps = 2000</code> y <code>lr = 0.002</code> sobre la característica centrada y confirma que la tasa pequeña termina por llegar.</p></div>

## 8. Sobreajuste: un adelanto

### 8.1 Más flexible no es mejor

El modelo lineal a menudo es incorrecto: las curvas dosis–respuesta se saturan, las señales oscilan. La <strong>regresión polinomial</strong> reemplaza \(x\) por \(x, x^2, \dots, x^d\) y ajusta un modelo lineal sobre ellas —lineal en los <em>parámetros</em>, de modo que las ecuaciones normales se aplican sin cambios, pero la curva puede doblarse—, y el grado \(d\) controla la flexibilidad. El experimento toma 20 puntos de entrenamiento de una curva seno más ruido, ajusta los grados 1, 3 y 15, y evalúa cada uno en los puntos de entrenamiento <em>y</em> en 200 puntos nuevos de la misma curva.



In [ ]:
# Regresión polinómica en un problema sintético 1-D: error de entrenamiento versus error en datos nuevos.
truth <- function(x) sin(2 * pi * x)
set.seed(0); xtr <- sort(runif(20)); ytr <- truth(xtr) + rnorm(20, 0, 0.3)
set.seed(1); xte <- sort(runif(200, min(xtr), max(xtr))); yte <- truth(xte) + rnorm(200, 0, 0.3)
xx <- seq(min(xtr), max(xtr), length = 400)
op <- par(mfrow = c(1, 3), mar = c(4, 4, 2.4, 1)); rows <- list()
for (k in seq_along(c(1, 3, 15))) { d <- c(1, 3, 15)[k]
  m <- lm(ytr ~ poly(xtr, d))
  tr_mse <- mean((predict(m) - ytr)^2); te_mse <- mean((predict(m, data.frame(xtr = xte)) - yte)^2)
  rows[[k]] <- data.frame(degree = d, parameters = d + 1, train_MSE = tr_mse, test_MSE = te_mse)
  plot(xtr, ytr, pch = 19, col = BLUE, cex = .8, ylim = c(-2, 2), xlab = "x", ylab = if (k==1) "y" else "",
       main = sprintf("degree %d: train %.3f, test %.3f", d, tr_mse, te_mse), cex.main = .85)
  lines(xx, truth(xx), col = GREY, lty = 2); lines(xx, predict(m, data.frame(xtr = xx)), col = ORANGE, lwd = 2) }
par(op); print(do.call(rbind, rows), row.names = FALSE)



<p class="caption"><b>Figura 9.</b> Tres ajustes polinomiales (naranja) a 20 puntos ruidosos (azul) de una curva seno (discontinua). El grado 1 no puede seguir la curva y se equivoca en ambos conjuntos: presenta <strong>subajuste</strong>. El grado 3 sigue la curva con el menor error de prueba, alrededor de 0.10, prácticamente la varianza del ruido \(0.3^2 = 0.09\), lo mejor que podría lograr cualquier modelo. El grado 15 pasa casi por los puntos de entrenamiento (MSE de entrenamiento 0.008), pero oscila bruscamente entre ellos; su error de prueba es enorme: presenta <strong>sobreajuste</strong>, pues aprendió el ruido de la muestra en lugar del patrón.</p>

### 8.2 El error de entrenamiento no es la cantidad de interés

La tabla enuncia la lección metodológica más importante del curso: <strong>el error de entrenamiento disminuye monótonamente con la flexibilidad y no sirve para elegir el modelo</strong>. Lo que importa es el error de <em>generalización</em> en datos no vistos, que baja, alcanza un mínimo en la flexibilidad adecuada y vuelve a subir. El Suplemento S1 desarrolla la metodología; aquí el objetivo es reconocer el fenómeno, porque todos los clasificadores de la Figura 4 tienen una perilla de flexibilidad —profundidad del árbol, número de vecinos, tamaño de la red— y todos pueden sobreajustar.

### 8.3 Regularización: regresión ridge

El grado 15 está subrestringido: nada impide que los coeficientes crezcan hasta valores absurdos para pasar por el ruido. La <strong>regularización</strong> agrega a la pérdida una penalización sobre el tamaño de los coeficientes, equilibrando ajuste y simplicidad. La <strong>regresión ridge</strong> (Hoerl y Kennard, 1970)<sup class="c"><a href="#r24">[24]</a></sup> es su forma más antigua:

\[ L_{\text{ridge}}(\boldsymbol\beta) = \lVert \mathbf{y} - \mathbf{X}\boldsymbol\beta \rVert^2 + \lambda \lVert \boldsymbol\beta \rVert^2 \;\Longrightarrow\; \hat{\boldsymbol\beta}_{\text{ridge}} = (\mathbf{X}^\top\mathbf{X} + \lambda \mathbf{I})^{-1} \mathbf{X}^\top \mathbf{y} \]

donde \(\lambda \ge 0\) fija la intensidad. La solución son las ecuaciones normales con \(\lambda\) en la diagonal —la "cresta" (ridge)—, lo que hace la matriz invertible incluso cuando \(\mathbf{X}^\top\mathbf{X}\) es singular y encoge todos los coeficientes hacia cero. El bloque de código aplica esta fórmula a las características de grado 15, estandarizadas para que la penalización trate por igual a todas las potencias.



In [ ]:
# Regresión ridge sobre las características de grado 15: las ecuaciones normales con lambda en la diagonal.
poly_design <- function(xv, d, lo, hi) { t <- 2 * (xv - lo) / (hi - lo) - 1; sapply(1:d, function(k) t^k) }
lo <- min(xtr); hi <- max(xtr); Phi <- poly_design(xtr, 15, lo, hi)
mu <- colMeans(Phi); sdv <- apply(Phi, 2, sd)
Ptr <- scale(Phi); Pte <- scale(poly_design(xte, 15, lo, hi), mu, sdv); Pxx <- scale(poly_design(xx, 15, lo, hi), mu, sdv)
plot(xtr, ytr, pch = 19, col = BLUE, cex = .8, ylim = c(-2, 2), xlab = "x", ylab = "y",
     main = "Ridge tames the degree-15 polynomial"); lines(xx, truth(xx), col = GREY, lty = 2)
rows <- list(); lcol <- c(ORANGE, GREEN, "black"); k <- 0
for (lam in c(0, 1e-4, 1e-2, 1, 100)) {
  b <- solve(t(Ptr) %*% Ptr + lam * diag(15), t(Ptr) %*% (ytr - mean(ytr)))
  pred <- function(P) P %*% b + mean(ytr)
  rows[[length(rows)+1]] <- data.frame(lambda = lam, train_MSE = mean((pred(Ptr) - ytr)^2),
    test_MSE = mean((pred(Pte) - yte)^2), coef_norm = sqrt(sum(b^2)))
  if (lam %in% c(0, 1e-2, 100)) { k <- k + 1; lines(xx, pred(Pxx), col = lcol[k], lwd = 2) } }
legend("topright", c("lambda = 0", "lambda = 0.01", "lambda = 100"), col = lcol, lty = 1, lwd = 2, bty = "n", cex = .8)
print(do.call(rbind, rows), row.names = FALSE)



<p class="caption"><b>Figura 10.</b> El polinomio de grado 15 con tres penalizaciones. \(\lambda = 0\) (naranja) es el sobreajuste de la Figura 9, con norma de coeficientes de alrededor de 11,000. \(\lambda = 0.01\) (verde): los mismos dieciséis parámetros producen una curva suave que sigue a la curva verdadera tan bien como el cúbico —MSE de prueba de alrededor de 0.12, norma de coeficientes 2.8, cuatro órdenes de magnitud menor—. \(\lambda = 100\) (negro): la penalización domina y la curva presenta subajuste. La intensidad de la regularización es una segunda perilla de flexibilidad, que se elige, igual que el grado, con datos reservados (Suplemento S1).</p>

<div class="box bridge"><h4>Puente hacia unidades posteriores y hacia QSAR</h4><p>Todo modelo tiene una perilla así: <code>C</code> en la regresión logística y en las máquinas de vectores de soporte (Unidad 3) es un \(\lambda\) inverso; el tamaño de hoja de un árbol (Unidad 4) y el decaimiento de pesos (weight decay) de una red (Unidad 6) cumplen el mismo papel. En QSAR (BGM324), con cientos de descriptores y unas pocas decenas de moléculas, \(p > n\), y ridge y sus parientes (lasso, elastic net, mínimos cuadrados parciales) son la única forma de ajustar siquiera un modelo lineal. Teoría: Hastie, Tibshirani y Friedman, capítulo 3.<sup class="c"><a href="#r26">[26]</a></sup></p></div>

## 9. Regresión logística: el primer clasificador

### 9.1 De un puntaje lineal a una probabilidad

El clasificador más simple reutiliza el modelo lineal: un puntaje \(z = \mathbf{x}^\top\boldsymbol\beta\), benigno si \(z > 0\), como en la Figura 3. Pero un puntaje no es una probabilidad, y ajustarlo por mínimos cuadrados a etiquetas 0 y 1 predice 1.4 o −0.2 y se ve arrastrado por casos que ya están lejos, en el lado correcto. La <strong>regresión logística</strong> pasa el puntaje por la <strong>sigmoide</strong>:

\[ p(\mathbf{x}) = P(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{x}^\top\boldsymbol\beta) = \frac{1}{1 + e^{-\mathbf{x}^\top\boldsymbol\beta}} \]

con \(\sigma(0) = 0.5\), \(\sigma(z) \to 1\) cuando \(z \to \infty\), y \(\sigma(-z) = 1 - \sigma(z)\). Invirtiendo: \(\mathbf{x}^\top\boldsymbol\beta = \log\frac{p}{1-p}\); el puntaje es el <strong>logaritmo de las odds</strong> (log-odds o logit), cada coeficiente es un cambio en log-odds por unidad de su característica, y \(e^{\beta_j}\) es una <em>razón de odds</em> (odds ratio), la cantidad que informa todo artículo clínico que usa el modelo. La frontera con umbral 0.5 sigue siendo el hiperplano \(\mathbf{x}^\top\boldsymbol\beta = 0\); la sigmoide agrega una afirmación calibrada de <em>qué tan lejos</em> de él está un caso. La formulación como regresión con ajuste por máxima verosimilitud es de Cox (1958).<sup class="c"><a href="#r27">[27]</a></sup>



In [ ]:
# La función sigmoide y su derivada.
sigmoid <- function(z) 1 / (1 + exp(-z)); z <- seq(-8, 8, length = 400)
plot(z, sigmoid(z), type = "l", col = BLUE, lwd = 2, xlab = "linear score z (log-odds)",
     ylab = "probability", main = "The sigmoid maps any score to a probability")
lines(z, sigmoid(z) * (1 - sigmoid(z)), col = ORANGE, lwd = 1.5, lty = 2)
abline(h = 0.5, col = GREY); abline(v = 0, col = GREY)
for (zz in c(-4, -2, 0, 2, 4)) { points(zz, sigmoid(zz), pch = 19, col = BLUE)
  text(zz + 0.6, sigmoid(zz) - 0.04, sprintf("%.3f", sigmoid(zz)), cex = .75) }
legend("left", c("sigmoid", "derivative s(z)(1-s(z))"), col = c(BLUE, ORANGE), lty = c(1, 2), bty = "n", cex = .8)



<p class="caption"><b>Figura 11.</b> La sigmoide y su derivada. Un puntaje de 0 corresponde a una probabilidad de 0.5; ±2 da 0.88 y 0.12; ±4 da 0.98 y 0.02. La derivada es máxima en la frontera y se anula lejos de ella, de modo que el entrenamiento lo impulsan los casos cercanos a la frontera, no los clasificados con confianza.</p>

### 9.2 La pérdida: log-loss

La <strong>máxima verosimilitud</strong> reemplaza a los mínimos cuadrados: se eligen los parámetros con los que las etiquetas observadas eran más probables. El conjunto de entrenamiento tenía probabilidad \(\prod_i p_i^{y_i}(1 - p_i)^{1 - y_i}\); se minimiza su logaritmo negativo, la <strong>pérdida logarítmica</strong> (log-loss, o entropía cruzada binaria):

\[ L(\boldsymbol\beta) = -\frac{1}{n}\sum_{i=1}^{n}\Big[\, y_i \log p_i + (1 - y_i)\log(1 - p_i) \,\Big], \qquad p_i = \sigma(\mathbf{x}_i^\top\boldsymbol\beta) \]

Un caso benigno (\(y_i = 1\)) predicho con \(p_i = 0.9\) aporta \(-\log 0.9 \approx 0.105\); con \(p_i = 0.1\), alrededor de 2.3; cuando \(p_i \to 0\), una penalización sin cota. Las predicciones erróneas hechas con confianza se castigan sin límite, lo que mantiene honestas las probabilidades. Su gradiente tiene la forma de la Sección 7.3 —\(\nabla L = \frac{1}{n}\mathbf{X}^\top(\mathbf{p} - \mathbf{y})\), características por residuos—, de modo que el descenso de gradiente se aplica sin cambios (Ejercicio 4), y la pérdida es convexa. Pero la sigmoide no es lineal, así que no hay forma cerrada: es el primer modelo cuyo entrenamiento <em>requiere</em> un optimizador (<code>glm</code> de R usa mínimos cuadrados iterativamente reponderados).

### 9.3 Ajuste del modelo con dos características

El bloque de código ajusta el modelo al grosor del grupo celular y al tamaño celular en la partición de entrenamiento de la Figura 4. La clase positiva es benigno (código 1).



In [ ]:
# Regresión logística con dos características: coeficientes, interpretación en log-odds, exactitud y log-loss.
dlog <- data.frame(thickness = bc$Cl.thickness, size = bc$Cell.size, y = y)
glm.fit <- glm(y ~ thickness + size, binomial, dlog[tr, ])
cf <- coef(glm.fit)
cat(sprintf("log-odds(benign) = %.2f + (%.3f) x thickness + (%.3f) x size\n", cf[1], cf[2], cf[3]))
cat(sprintf("odds ratio per 1 unit of thickness: %.3f | per 1 unit of size: %.3f\n", exp(cf[2]), exp(cf[3])))
p_te <- predict(glm.fit, dlog[-tr, ], type = "response"); y_te <- dlog$y[-tr]
logloss <- function(p, yy) { p <- pmin(pmax(p, 1e-12), 1 - 1e-12); -mean(yy * log(p) + (1 - yy) * log(1 - p)) }
cat(sprintf("test accuracy (threshold 0.5) = %.3f | test log-loss = %.3f | base-rate log-loss = %.3f\n",
            mean((p_te > 0.5) == (y_te == 1)), logloss(p_te, y_te),
            logloss(rep(mean(dlog$y[tr]), length(y_te)), y_te)))



Ambos coeficientes son negativos: grupos celulares más gruesos y núcleos más grandes reducen el log-odds de benignidad. Cada unidad adicional de grosor del grupo celular multiplica las odds de benignidad por alrededor de 0.53, es decir, aproximadamente las reduce a la mitad, manteniendo todo lo demás constante. La exactitud de prueba con umbral 0.5 es 0.927, la de las rectas de la Figura 3, como se espera de otra frontera recta. Lo nuevo es la log-loss, 0.145 frente a 0.625 al predecir la tasa base: cuánto mejores son las probabilidades, algo que la exactitud no puede ver.

### 9.4 Ejemplo resuelto a mano: una probabilidad

Redondea los coeficientes a \(\beta_0 = 7.32\), \(\beta_{\text{thickness}} = -0.63\), \(\beta_{\text{size}} = -1.23\) y toma un aspirado nuevo con grosor del grupo celular 5 y tamaño celular 3.

**Paso 1 — puntaje.** \(z = 7.32 - 0.63 \times 5 - 1.23 \times 3 = 7.32 - 3.15 - 3.69 = 0.48\).

**Paso 2 — sigmoide.** \(e^{-0.48} \approx 0.619\), de modo que \(p(\text{benign}) = 1/(1 + 0.619) \approx 0.618\) y \(p(\text{malignant}) \approx 0.382\).

**Paso 3 — decisión.** Con umbral 0.5 el caso es benigno, pero el modelo dice que hay casi dos probabilidades en cinco de malignidad, una afirmación que llevaría a un clínico a actuar. Un segundo caso, grosor 8 y tamaño 8: \(z = -7.56\), \(p(\text{benign}) \approx 0.001\).



In [ ]:
# Confirmación del cálculo a mano con el modelo ajustado (sin redondear) y con los coeficientes redondeados.
new_cases <- data.frame(thickness = c(5, 8), size = c(3, 8))
z_hand <- 7.32 - 0.63 * new_cases$thickness - 1.23 * new_cases$size
cat("rounded coefficients : z =", round(z_hand, 3), " p(benign) =", round(sigmoid(z_hand), 3), "\n")
cat("fitted model         : z =", round(predict(glm.fit, new_cases), 3),
    " p(benign) =", round(predict(glm.fit, new_cases, type = "response"), 3), "\n")



Los coeficientes redondeados y los exactos coinciden hasta el segundo decimal. El modelo entrega la probabilidad; la etiqueta es un paso aparte (Sección 9.5).



In [ ]:
# La superficie de probabilidad del modelo ajustado, su frontera y los 205 casos de prueba.
gx <- seq(0, 11, length = 200); gy <- seq(0, 11, length = 200)
grid <- expand.grid(thickness = gx, size = gy)
Pg <- matrix(predict(glm.fit, grid, type = "response"), 200)
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
image(gx, gy, Pg, col = hcl.colors(20, "Blue-Red", rev = TRUE), xlab = "clump thickness", ylab = "cell size",
      main = "P(benign): solid line P = 0.5"); contour(gx, gy, Pg, levels = c(0.1, 0.5, 0.9), add = TRUE, lwd = c(.7, 1.6, .7))
points(dlog$thickness[-tr], dlog$size[-tr], pch = 19, cex = .4, col = ifelse(y_te == 1, BLUE, ORANGE))
points(new_cases, pch = 8, col = GREEN, cex = 2)
hist(p_te[y_te == 1], breaks = seq(0, 1, .05), col = adjustcolor(BLUE, .7), xlab = "predicted P(benign)",
     main = "Predicted probabilities by true class", ylim = c(0, 60))
hist(p_te[y_te == 0], breaks = seq(0, 1, .05), col = adjustcolor(ORANGE, .7), add = TRUE)
abline(v = 0.5); legend("top", c("true benign", "true malignant"), fill = c(BLUE, ORANGE), bty = "n", cex = .8)
par(op)



<p class="caption"><b>Figura 12.</b> Izquierda: probabilidad predicha de benignidad sobre el plano de características, la frontera \(P = 0.5\) (línea continua) y las líneas \(P = 0.1\) y \(0.9\). La frontera es recta; los colores agregan la franja de incertidumbre donde se encuentran casi todos los casos mal clasificados. Estrellas: los casos resueltos a mano, uno con \(P \approx 0.62\) y otro en plena zona roja. Derecha: probabilidades predichas de los casos de prueba según la clase verdadera —la mayoría de los benignos cerca de 1, la mayoría de los malignos cerca de 0, y un solapamiento en el centro donde están los casos que cualquier umbral clasificará mal—.</p>

### 9.5 El umbral y el costo de los errores

El histograma es todo el problema de decisión. Esté donde esté el umbral, los casos malignos a su derecha se declaran benignos (<strong>falsos negativos</strong>, cánceres no detectados) y los benignos a su izquierda, malignos (<strong>falsos positivos</strong>, biopsias innecesarias); moverlo cambia uno por otro. El valor por defecto de 0.5 solo es correcto si ambos errores cuestan lo mismo. Si un cáncer no detectado cuesta diez veces lo que una biopsia, la regla óptima declara maligno siempre que \(P(\text{malignant}) > 1/11 \approx 0.09\). El bloque de código calcula la sensibilidad, la especificidad y el costo total con una razón 10 : 1 para cada umbral.



In [ ]:
# Sensibilidad, especificidad y costo total en función del umbral sobre P(maligno).
p_mal <- 1 - p_te; is_mal <- (y_te == 0); ths <- seq(0.02, 0.98, 0.01)
tab <- data.frame(t(sapply(ths, function(t) {
  call_mal <- p_mal >= t
  tp <- sum(call_mal & is_mal); fn <- sum(!call_mal & is_mal)
  fp <- sum(call_mal & !is_mal); tn <- sum(!call_mal & !is_mal)
  c(threshold = t, sensitivity = tp / sum(is_mal), specificity = tn / sum(!is_mal),
    missed = fn, biopsies = fp, cost_10 = 10 * fn + fp) })))
best <- tab[which.min(tab$cost_10), ]
cat(sprintf("Of %d test cases (%d malignant): lowest 10:1 cost at threshold %.2f (%d missed, %d unnecessary biopsies, cost %d)\n",
            length(y_te), sum(is_mal), best$threshold, best$missed, best$biopsies, best$cost_10))
at05 <- tab[which.min(abs(tab$threshold - 0.5)), ]
cat(sprintf("At threshold 0.5: %d missed, %d unnecessary biopsies\n", at05$missed, at05$biopsies))
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(tab$threshold, tab$sensitivity, type = "l", col = ORANGE, lwd = 2, ylim = c(0, 1),
     xlab = "threshold on P(malignant)", ylab = "rate", main = "The two error types trade off")
lines(tab$threshold, tab$specificity, col = BLUE, lwd = 2); abline(v = c(0.5, best$threshold), col = c(GREY, GREEN), lty = c(3, 2))
legend("bottom", c("sensitivity", "specificity"), col = c(ORANGE, BLUE), lty = 1, bty = "n", cex = .8)
plot(tab$threshold, tab$cost_10, type = "l", lwd = 2, xlab = "threshold on P(malignant)",
     ylab = "total cost on 205 test cases", main = "Total cost under two cost ratios")
lines(tab$threshold, tab$missed + tab$biopsies, col = GREY, lty = 2)
abline(v = c(0.5, best$threshold), col = c(GREY, GREEN), lty = c(3, 2))
legend("top", c("10:1 cost", "1:1 cost"), col = c("black", GREY), lty = c(1, 2), bty = "n", cex = .8)
par(op)



<p class="caption"><b>Figura 13.</b> Izquierda: sensibilidad y especificidad en los 205 casos de prueba a medida que el umbral sobre \(P(\text{malignant})\) se mueve de 0 a 1. Derecha: costo total con dos razones de costo. Con costos iguales (línea discontinua) el mínimo está cerca de 0.5; si un cáncer no detectado pesa diez veces lo que una biopsia (línea continua), el mínimo se desplaza muy a la izquierda, a alrededor de 0.05 (verde), donde se detectan todos los cánceres a costa de biopsias adicionales, frente a varios no detectados con 0.5. Nada del <em>modelo</em> cambió entre los paneles; solo cambió la regla de decisión.</p>

<div class="box key"><h4>Modelo, probabilidad, decisión</h4><p>Un clasificador probabilístico separa el <em>modelo</em> (ajustado una vez), la <em>probabilidad</em> (lo que sabe sobre un caso) y la <em>decisión</em> (que depende de costos clínicos y éticos, no estadísticos). El análisis de curvas de decisión formaliza la elección del umbral como beneficio clínico neto,<sup class="c"><a href="#r28">[28]</a></sup> y la calibración —si los casos a los que se asigna \(P = 0.3\) son malignos el 30 % de las veces— debe verificarse por separado de la exactitud, y se descuida en la mayoría de los modelos publicados.<sup class="c"><a href="#r29">[29]</a></sup> El Suplemento S1 desarrolla ambos temas.</p></div>

### 9.6 La regresión logística en medicina

Los puntajes de riesgo de la práctica diaria son regresiones logísticas disfrazadas. El <strong>puntaje de riesgo de Framingham</strong> para enfermedad coronaria se ajustó con 5,345 participantes seguidos durante 12 años; sus tablas de puntos son coeficientes redondeados a enteros para poder calcular en papel.<sup class="c"><a href="#r30">[30]</a></sup> El <strong>SAPS II</strong>, para la mortalidad en cuidados intensivos, es una regresión logística sobre 12,997 pacientes de 137 unidades, con sus 17 variables seleccionadas y ponderadas por la regresión.<sup class="c"><a href="#r31">[31]</a></sup> El <strong>modelo de Gail</strong> entrega la probabilidad de cáncer de mama de una mujer a partir de su edad y sus antecedentes reproductivos y familiares.<sup class="c"><a href="#r32">[32]</a></sup> Cada uno es un puntaje lineal en características clínicas, una sigmoide y un umbral que es una decisión clínica. La declaración TRIPOD especifica cómo deben informarse estos modelos —incluida la ecuación completa— y es la lista de verificación que debes aplicar cuando leas uno.<sup class="c"><a href="#r33">[33]</a></sup>

### 9.7 Más de dos clases

<strong>Uno contra el resto</strong> (one-versus-rest) ajusta \(K\) modelos binarios y elige el que tiene más confianza. La <strong>regresión multinomial (softmax)</strong> ajusta un puntaje \(z_k = \mathbf{x}^\top\boldsymbol\beta_k\) por clase y los convierte conjuntamente con la <strong>softmax</strong>:

\[ P(y = k \mid \mathbf{x}) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}} \]

valores positivos que suman uno y que, para \(K = 2\), se reducen a la sigmoide de la diferencia de puntajes. La pérdida es la misma log-loss, \(-\frac{1}{n}\sum_i \log P(y = y_i \mid \mathbf{x}_i)\), y la capa de salida de todo clasificador basado en redes neuronales (Unidades 6 y 8) es una softmax. La edición en Python la aplicó a los dígitos manuscritos de UCI<sup class="c"><a href="#r34">[34]</a></sup> (diez clases, 64 píxeles); R no incluye ese conjunto de datos, así que el bloque de código <em>simula</em> cuatro clases de parches de imagen de 8×8 —una barra vertical, una barra horizontal, una diagonal y un bloque central—, cada una una plantilla con ruido, aplanada en 64 características de píxel, y ajusta <code>nnet::multinom</code>.<sup class="c"><a href="#r38">[38]</a></sup>



In [ ]:
# Regresión logística multinomial (softmax) sobre parches de imagen simulados de 8x8: cuatro clases, 64 píxeles.
set.seed(0); amp <- 1.6; sig <- 1.5
gen <- function(cl) { m <- matrix(rnorm(64, 0, sig), 8, 8)
  if (cl == 1) m[, 4:5] <- m[, 4:5] + amp        # barra vertical
  if (cl == 2) m[4:5, ] <- m[4:5, ] + amp        # barra horizontal
  if (cl == 3) for (k in 1:8) m[k, k] <- m[k, k] + amp   # diagonal
  if (cl == 4) m[3:6, 3:6] <- m[3:6, 3:6] + amp  # bloque central
  as.vector(m) }
N <- 200; lab <- rep(1:4, each = N / 4); Xi <- t(sapply(lab, gen))
set.seed(0); tri <- sample(N, round(0.7 * N))
sm <- multinom(factor(lab[tri]) ~ ., data.frame(Xi[tri, ]), trace = FALSE, decay = 0.2, MaxNWts = 5000)
prob <- predict(sm, data.frame(Xi[-tri, ]), type = "probs"); pr <- predict(sm, data.frame(Xi[-tri, ]))
cat(sprintf("test accuracy on %d patches = %.3f | probabilities sum to one: %s | mean confidence = %.3f\n",
            N - length(tri), mean(pr == lab[-tri]), all(abs(rowSums(prob) - 1) < 1e-8), mean(apply(prob, 1, max))))
cm <- table(true = lab[-tri], predicted = pr)
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
image(1:8, 1:8, matrix(gen(3), 8), col = grey.colors(20), xlab = "", ylab = "", main = "one simulated 'diagonal' patch")
image(1:4, 1:4, t(cm[4:1, ]), col = hcl.colors(12, "Blues", rev = TRUE), xlab = "predicted class",
      ylab = "true class", axes = FALSE, main = "Confusion matrix (60 test patches)")
axis(1, 1:4); axis(2, 1:4, labels = 4:1)
for (i in 1:4) for (j in 1:4) text(j, 5 - i, cm[i, j], col = ifelse(cm[i, j] > max(cm)/2, "white", "black"))
par(op)



<p class="caption"><b>Figura 14.</b> Derecha: matriz de confusión en 60 parches simulados reservados; filas, clase verdadera; columnas, predicha; diagonal, aciertos. La exactitud es 0.95 —el modelo softmax lineal lee mal alrededor de un parche de cada veinte— y las entradas fuera de la diagonal muestran <em>qué</em> confusiones ocurren (la plantilla diagonal leída ocasionalmente como el bloque central), información que la exactitud oculta y que importa cuando algunas confusiones son más graves que otras. Izquierda: un parche "diagonal" con ruido, para mostrar cuánto ruido logra atravesar el modelo.</p>

<div class="box key"><h4>Ejercicio 4</h4><p>(a) Implementa el descenso de gradiente para la regresión logística sobre las dos características estandarizadas: desde \(\boldsymbol\beta = \mathbf{0}\), gradiente \(\frac{1}{n}\mathbf{X}^\top(\sigma(\mathbf{X}\boldsymbol\beta) - \mathbf{y})\), tasa 0.5, 500 pasos; compara con <code>glm</code> sobre las mismas características. (b) Cambia la razón de costos en el bloque de código del umbral a 3 y a 30. (c) Aumenta el ruido de las imágenes <code>sig</code> a 2.5 y observa cómo se llena la matriz de confusión.</p></div>

## 10. Adelantos de los otros paradigmas

### 10.1 No supervisado: k-means sobre las características de cáncer de mama, sin etiquetas

<strong>k-means</strong> divide \(n\) puntos en \(k\) grupos minimizando las distancias al cuadrado a los centroides de los grupos, mediante una alternancia que vale la pena seguir una vez a mano: (1) <em>asignar</em> cada punto al centroide más cercano; (2) <em>actualizar</em> cada centroide a la media de sus puntos; repetir hasta que nada cambie (Lloyd, 1957; MacQueen, 1967).<sup class="c"><a href="#r20">[20]</a></sup><sup class="c"><a href="#r19">[19]</a></sup> Es el clasificador de centroide más cercano de la Unidad 1 ejecutado <em>sin etiquetas</em>: adivina los grupos, calcula los centroides, vuelve a adivinar, itera. El bloque de código lo ejecuta con \(k = 2\) sobre las dos características —sin ver nunca el diagnóstico— y solo después lo compara con las etiquetas.



In [ ]:
# k-means (k = 2) sobre dos características de los 683 aspirados, con las etiquetas ocultas; luego se compara con las etiquetas.
set.seed(0); km <- kmeans(X2, centers = 2, nstart = 10)          # solo X2: sin etiquetas
clusters <- km$cluster - 1
agreement <- max(mean(clusters == yv), mean(clusters != yv))     # la numeración de los clústeres es arbitraria
ct <- table(cluster = clusters, biopsy = ifelse(yv == 0, "malignant", "benign"))
cat("centroids found without labels:\n"); print(round(km$centers, 2))
cat(sprintf("agreement between clusters and diagnosis (best label matching): %.3f = %d of %d\n",
            agreement, round(agreement * nrow(bc)), nrow(bc)))
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
plot(X2, col = ifelse(clusters == round(mean(clusters[yv == 1])), BLUE, ORANGE), pch = 19, cex = .4,
     xlab = "clump thickness", ylab = "cell size", main = "What k-means found (no labels)")
points(km$centers, pch = 4, cex = 2, lwd = 3)
plot(X2, col = ifelse(yv == 1, BLUE, ORANGE), pch = 19, cex = .4, xlab = "clump thickness", ylab = "cell size",
     main = "The biopsy labels (blue benign)")
par(op); print(ct)



<p class="caption"><b>Figura 15.</b> Izquierda: los dos grupos encontrados solo a partir de las características, con sus centroides (cruces). Derecha: los mismos puntos coloreados según la biopsia. El algoritmo no supervisado recuperó una partición que concuerda con el patólogo en alrededor del 93 % de los aspirados (633 de 683), porque los dos diagnósticos resultan formar dos regiones densas y k-means encuentra regiones densas; la tabla cruzada muestra los casos que quedaron en el lado equivocado de la separación.</p>

Es alentador y, a la vez, una advertencia. La estructura en datos no etiquetados puede coincidir con categorías clínicas: es la esperanza detrás de toda búsqueda de subtipos de pacientes. Pero la concordancia se acerca a la de los métodos supervisados solo porque aquí las clases se separan limpiamente; al algoritmo se le dijo que existían <em>dos</em> grupos, y si los grupos naturales hubieran sido tumores "grandes" y "pequeños", habría encontrado esos y nada en su resultado lo habría indicado. La Unidad 5 trata la elección de \(k\) y la validación de un agrupamiento.

### 10.2 Refuerzo: un pasillo de cinco estados

Posiciones 0 a 4; el agente parte en 2 y se mueve a la izquierda o a la derecha. La posición 0 es una salida con recompensa 0.3 y la posición 4, una salida con recompensa 1.0; ambas terminan el episodio; los demás movimientos no reciben nada. A un paso de la salida pequeña, el agente enfrenta la pregunta que define el problema: ¿tomar 0.3 ahora o caminar tres pasos por 1.0? La respuesta depende del <strong>factor de descuento</strong> \(\gamma \in [0, 1)\): una recompensa a \(t\) pasos de distancia vale \(\gamma^t\) de una inmediata. El <strong>valor</strong> \(V^*(s)\), la mejor suma descontada de recompensas futuras desde \(s\), satisface la ecuación de Bellman:

\[ V^*(s) = \max_{a} \Big[\, r(s, a) + \gamma\, V^*(s') \,\Big] \]

donde \(s'\) es el estado al que lleva la acción \(a\), y \(V^*\) de una salida es 0. La <strong>iteración de valores</strong> parte con \(V = 0\) y aplica el lado derecho repetidamente hasta que \(V\) deja de cambiar; la política óptima toma, en cada estado, la acción que maximiza.<sup class="c"><a href="#r11">[11]</a></sup>

**Resuelto a mano, \(\gamma = 0.9\).** Hay que calcular los valores de las posiciones 1, 2 y 3; se parte con \(V = (0, 0, 0)\).

| barrido | \(V(1) = \max(0.3,\; 0.9\,V(2))\) | \(V(2) = \max(0.9\,V(1),\; 0.9\,V(3))\) | \(V(3) = \max(0.9\,V(2),\; 1.0)\) |
|---|---|---|---|
| 0 | 0 | 0 | 0 |
| 1 | \(\max(0.3, 0) = 0.3\) | \(\max(0, 0) = 0\) | \(\max(0, 1.0) = 1.0\) |
| 2 | \(\max(0.3, 0) = 0.3\) | \(\max(0.27, 0.9) = 0.9\) | \(1.0\) |
| 3 | \(\max(0.3, 0.81) = 0.81\) | \(0.9\) | \(1.0\) |
| 4 | \(0.81\) | \(0.9\) | \(1.0\) — convergió |

\(V^* = (0.81, 0.9, 1.0)\) y la acción que maximiza es "derecha" en todas partes: el agente ignora la salida pequeña incluso estando junto a ella, porque \(0.9^2 = 0.81 > 0.3\). Con \(\gamma = 0.5\), la posición 1 da \(\max(0.3, 0.25) = 0.3\) y el agente toma la salida pequeña: un agente miope se conforma con menos.



In [ ]:
# Iteración de valores en el pasillo de cinco estados, para dos factores de descuento.
step <- function(s, a) { s2 <- s + a                               # moverse; devuelve (nuevo estado, recompensa, terminado)
  if (s2 == 0) return(list(s2, 0.3, TRUE)); if (s2 == 4) return(list(s2, 1.0, TRUE)); list(s2, 0, FALSE) }
value_iteration <- function(gamma, tol = 1e-9) {
  V <- rep(0, 5); sweeps <- 0
  repeat { Vn <- V
    for (s in c(1, 2, 3)) Vn[s + 1] <- max(sapply(c(-1, 1), function(a) {
      r <- step(s, a); r[[2]] + gamma * (if (r[[3]]) 0 else V[r[[1]] + 1]) }))
    sweeps <- sweeps + 1; if (max(abs(Vn - V)) < tol) return(list(V = Vn, sweeps = sweeps)); V <- Vn } }
for (gamma in c(0.9, 0.5)) { out <- value_iteration(gamma)
  policy <- sapply(c(1, 2, 3), function(s) {
    rR <- step(s, 1); rL <- step(s, -1)
    vR <- rR[[2]] + gamma * (if (rR[[3]]) 0 else out$V[rR[[1]] + 1])
    vL <- rL[[2]] + gamma * (if (rL[[3]]) 0 else out$V[rL[[1]] + 1]); if (vR >= vL) "right" else "left" })
  cat(sprintf("gamma = %.1f: V*(1..3) = %s after %d sweeps | policy at 1,2,3: %s\n",
              gamma, paste(round(out$V[2:4], 3), collapse = ", "), out$sweeps, paste(policy, collapse = ", "))) }



La iteración de valores reproduce \(V^* = (0.81, 0.9, 1.0)\) para \(\gamma = 0.9\); para \(\gamma = 0.5\), el agente en la posición 1 gira a la izquierda. Pero necesita conocer de antemano las reglas del entorno (<code>step</code>). El <strong>Q-learning</strong> aprende una tabla \(Q(s, a)\) solo a partir de la experiencia, probando acciones (explorando al azar con probabilidad \(\epsilon\)) y actualizando

\[ Q(s, a) \leftarrow Q(s, a) + \alpha\,\Big[\, r + \gamma \max_{a'} Q(s', a') - Q(s, a) \,\Big], \]

donde el corchete es el <em>error de diferencia temporal</em> entre lo esperado y lo observado, y \(\alpha\) es una tasa de aprendizaje como en el descenso de gradiente. Watkins y Dayan demostraron que la tabla converge a \(Q^*\), con \(V^*(s) = \max_a Q^*(s, a)\).<sup class="c"><a href="#r35">[35]</a></sup>



In [ ]:
# Q-learning: los mismos valores aprendidos solo a partir de la experiencia, sin acceso a las reglas.
set.seed(0); gamma <- 0.9; alpha <- 0.5; epsilon <- 0.3
Q <- matrix(0, 5, 2); V_star <- value_iteration(gamma)$V; errors <- numeric(600)
for (episode in 1:600) { s <- 2                                   # cada episodio comienza en el centro
  for (t in 1:50) {
    a <- if (runif(1) < epsilon) sample(2, 1) else which.max(Q[s + 1, ])   # explorar o explotar
    r <- step(s, c(-1, 1)[a]); target <- if (r[[3]]) r[[2]] else r[[2]] + gamma * max(Q[r[[1]] + 1, ])
    Q[s + 1, a] <- Q[s + 1, a] + alpha * (target - Q[s + 1, a]); s <- r[[1]]; if (r[[3]]) break }
  errors[episode] <- max(abs(apply(Q[2:4, ], 1, max) - V_star[2:4])) }
cat("learned Q(s,a) for positions 1..3 [left, right]:\n"); print(round(Q[2:4, ], 3))
cat("V from Q =", round(apply(Q[2:4, ], 1, max), 3), "| value iteration =", round(V_star[2:4], 3), "\n")
plot(errors, type = "l", col = BLUE, log = "y", xlab = "episode", ylab = "max |V from Q - V*|",
     main = "Q-learning converges to the values from value iteration")



<p class="caption"><b>Figura 16.</b> Mayor error entre los valores implicados por la tabla Q y \(V^*\) a lo largo de 600 episodios. Solo por ensayo y error, el agente descubre los valores que la iteración de valores calculó a partir de las reglas y, por lo tanto, la misma política: en todas las posiciones, la entrada "derecha" de la tabla supera a la entrada "izquierda".</p>

Nada en el bloque de código de Q-learning conoce las recompensas ni las transiciones; llama a <code>step</code> y observa. Ese es el sentido en que "refuerzo" nombra un paradigma: la experiencia la generan las acciones del aprendiz, el desempeño es la recompensa acumulada y la tarea es una política. Escala el pasillo al estado de un paciente en cuidados intensivos, las acciones a dosis y la recompensa a la supervivencia, y tendrás el "AI Clinician" de la Sección 3.4<sup class="c"><a href="#r12">[12]</a></sup>, con sus dificultades: experiencia tomada de registros de lo que hicieron los clínicos, un estado que nunca se observa por completo y una recompensa tan diferida como puede serlo una retroalimentación.

<div class="box key"><h4>Ejercicio 5</h4><p>(a) Fija <code>epsilon = 0</code> en el bloque de código de Q-learning. ¿Sigue encontrando el agente la salida correcta? Explícalo pensando en los primeros episodios. (b) Fija la recompensa izquierda en 0.85 con \(\gamma = 0.9\). ¿Qué salida debería tomar el agente desde las posiciones 1 y 2? Confírmalo con la iteración de valores.</p></div>

## 11. Adelanto: cómo sabremos que un modelo funciona

Todos los experimentos anteriores midieron un modelo en datos no vistos, y la Sección 8 mostró por qué. El Suplemento S1 desarrolla la metodología; esta sección fija su vocabulario.

**Conjuntos de entrenamiento, validación y prueba.** Los datos etiquetados se dividen al azar en un <strong>conjunto de entrenamiento</strong>, en el que se ajustan los parámetros; un <strong>conjunto de validación</strong>, en el que se eligen las perillas de flexibilidad (grado, \(\lambda\), profundidad del árbol, el umbral de la Sección 9.5) comparando candidatos; y un <strong>conjunto de prueba</strong>, que se usa una sola vez al final para informar el desempeño del modelo final en datos que no influyeron en ninguna de sus decisiones. Las proporciones habituales son 60/20/20, 70/15/15 y 80/10/10 por ciento, <em>estratificadas</em> para conservar las proporciones de clases. Con pocos datos, la <strong>validación cruzada de k particiones</strong> (k-fold) reemplaza al conjunto de validación: se dividen los datos de entrenamiento en \(k\) particiones (5 o 10), se ajusta \(k\) veces con \(k - 1\) particiones, se evalúa en la restante y se promedia; la versión estratificada conserva las proporciones de clases, y dejar uno fuera (leave-one-out) corresponde a \(k = n\).

El **sobreajuste** es un modelo que, además del patrón, aprendió los accidentes de su muestra —ruido, valores atípicos, correlaciones espurias—: error de entrenamiento bajo y error alto en datos nuevos, como el polinomio de grado 15; crece con la flexibilidad, con menos casos y con etiquetas más ruidosas, y solo se detecta en datos no vistos. El **subajuste** es un modelo demasiado rígido para el patrón, con error alto en ambos, como la recta ajustada al seno. La descomposición sesgo–varianza del Suplemento S1 explica por qué el error de generalización tiene forma de U entre ambos extremos.

**Métricas para un clasificador.** La <strong>matriz de confusión</strong> (Figura 14) tabula las clases verdaderas frente a las predichas; con dos clases, sus celdas son verdaderos positivos (TP), falsos positivos (FP), falsos negativos (FN) y verdaderos negativos (TN). <strong>Exactitud</strong> = (TP + TN)/n, engañosa con clases desbalanceadas; <strong>precisión</strong> = TP/(TP + FP), la fracción de predicciones positivas que son correctas; <strong>sensibilidad</strong> (recall) = TP/(TP + FN), la fracción de positivos verdaderos detectados; <strong>especificidad</strong> = TN/(TN + FP); <strong>F1</strong>, la media armónica de precisión y sensibilidad; <strong>AUC-ROC</strong>, el área bajo la curva de sensibilidad frente a 1 − especificidad a medida que se mueve el umbral (las dos curvas de la Figura 13 graficadas una contra otra), que mide qué tan bien ordena el modelo los casos con independencia de cualquier umbral: 0.5 para una moneda, 1.0 para un orden perfecto. En regresión, las métricas análogas son MSE, RMSE y \(R^2\) en datos reservados. Las clases desbalanceadas, los umbrales, la calibración, los intervalos de confianza y la fuga de información entre particiones se desarrollan en el Suplemento S1, que debe leerse antes de la Unidad 3.

## 12. Resumen

Ahora deberías poder explicar, sin apuntes:

- La \(T\), la \(P\) y la \(E\) de Mitchell; el aprendizaje como inducción y lo que eso implica respecto de la certeza.
- Los siete pasos de un proyecto de ciencia de datos y por qué el aprendizaje automático es uno de ellos.
- Modelo, pérdida, optimizador; los paradigmas según su retroalimentación (supervisado, no supervisado con sus cuatro subtipos, por refuerzo, semisupervisado y autosupervisado); clase frente a variable objetivo.
- Los enfoques estadístico, neuronal y evolutivo como búsquedas en un mismo espacio de hipótesis; clasificador y frontera de decisión; por qué k-means no es un clasificador.
- Mínimos cuadrados, las ecuaciones normales \(\mathbf{X}^\top\mathbf{X}\boldsymbol\beta = \mathbf{X}^\top\mathbf{y}\), el ajuste de cuatro puntos a mano, coeficientes, \(R^2\), residuos.
- Descenso de gradiente: actualización, regímenes de la tasa de aprendizaje, condicionamiento y escalamiento, variantes estocásticas.
- Sobreajuste, subajuste y lo que ridge agrega a las ecuaciones normales.
- Regresión logística: sigmoide, log-odds, log-loss, una probabilidad a mano, el umbral como decisión.
- Qué calculan k-means, la iteración de valores y el Q-learning, y a partir de qué experiencia.
- Entrenamiento/validación/prueba, validación cruzada, matriz de confusión, exactitud, precisión, sensibilidad, F1, AUC.

## 13. Autoevaluación

<details><summary>1. Un hospital quiere marcar, al momento del ingreso, a los pacientes que probablemente necesitarán cuidados intensivos dentro de 48 horas, a partir de sus primeros signos vitales y resultados de laboratorio. Identifica \(T\), \(P\) y \(E\) y el paradigma.</summary>\(T\): asignar "UCI dentro de 48 h" o no a un ingreso a partir de sus mediciones iniciales; \(P\): p. ej., sensibilidad a una tasa fija de falsas alarmas; \(E\): ingresos anteriores con su desenlace. Clasificación supervisada.</details>

<details><summary>2. ¿Inductiva o deductiva?: (a) "La calculadora de dosis aplica a este paciente la regla basada en el peso"; (b) "En 2,000 pacientes anteriores esta dosis fue tóxica cuando el aclaramiento de creatinina era inferior a 30 mL/min, así que la regla debería incluir un ajuste renal".</summary>(a) Deducción: una regla conocida aplicada a un caso. (b) Inducción: una regla general propuesta a partir de casos, posiblemente errónea para casos distintos de los observados.</details>

<details><summary>3. "Los datos ya están en la base de datos, así que podemos empezar a ajustar modelos mañana." ¿Qué pasos del protocolo se omiten y qué podría salir mal en cada uno?</summary>Identificación del problema (variable objetivo sin definir: se responde la pregunta equivocada), limpieza (valores faltantes codificados como ceros, pacientes duplicados, identificadores usados como características) y exploración (una característica que es un sustituto de la etiqueta, errores de digitación tomados como valores extremos).</details>

<details><summary>4. En la Figura 3, dos métodos produjeron rectas similares, y LDA una tercera cercana a ellas. ¿Son, por lo tanto, equivalentes los tres enfoques?</summary>No. Compartían el espacio de hipótesis (rectas), así que ninguno podía encontrar otra cosa; difieren en cómo buscan, en qué más aportan y, con espacios más ricos, en lo que pueden representar.</details>

<details><summary>5. Ajusta a mano una recta de mínimos cuadrados a \((1, 2), (2, 3), (3, 5), (4, 6)\).</summary>\(\bar{x} = 2.5\), \(\bar{y} = 4\). Productos de las desviaciones: \(3 + 0.5 + 0.5 + 3 = 7\); \(\sum (x - \bar{x})^2 = 5\). Pendiente \(1.4\), intercepto \(4 - 1.4 \times 2.5 = 0.5\): \(\hat{y} = 0.5 + 1.4x\); residuos \(0.1, -0.3, 0.3, -0.1\), que suman cero.</details>

<details><summary>6. El coeficiente de las habitaciones era 9.1 solo y 3.8 con trece características. ¿Cuál es "el efecto de las habitaciones"?</summary>Ninguno, en sentido causal. El primero es la diferencia promedio por habitación; el segundo, la diferencia entre sectores censales iguales en las otras doce variables, que absorben parte de la asociación.</details>

<details><summary>7. El descenso de gradiente divergió con tasa 0.021 y convergió con 0.015. ¿Qué fija el límite y qué cambio en los datos lo lleva a 1.0?</summary>El mayor valor propio de la hessiana, \(\lambda_{\max} \approx 102\): la convergencia requiere \(\eta < 2/\lambda_{\max} \approx 0.0196\). Estandarizar la característica hace que la hessiana del MSE sea \(2\mathbf{I}\), de modo que el límite pasa a ser \(\eta < 1\).</details>

<details><summary>8. En la Figura 9, el grado 15 tiene el menor error de entrenamiento y el mayor error de prueba. Explica ambos hechos y qué cambió ridge en la Figura 10.</summary>Dieciséis parámetros pasan cerca de veinte puntos (error de entrenamiento pequeño); entre ellos, la curva sigue el ruido (error de prueba grande). Ridge sumó \(\lambda\mathbf{I}\) a \(\mathbf{X}^\top\mathbf{X}\), lo que encogió los coeficientes en órdenes de magnitud y forzó una curva suave.</details>

<details><summary>9. Con \(z = 7.32 - 0.63\,t - 1.23\,s\) (grosor \(t\), tamaño \(s\)), calcula \(P(\text{benign})\) para grosor 6 y tamaño 2, y la razón de odds por unidad de \(s\).</summary>\(z = 7.32 - 3.78 - 2.46 = 1.08\); \(P = 1/(1 + e^{-1.08}) \approx 0.746\), de modo que \(P(\text{malignant}) \approx 0.254\). Razón de odds \(e^{-1.23} \approx 0.29\) por unidad de tamaño celular.</details>

<details><summary>10. ¿Por qué el umbral óptimo de la Figura 13 quedó muy por debajo de 0.5 con una razón de costos 10 : 1, y cuál es en general el umbral óptimo según los costos?</summary>Bajar el umbral convierte falsos negativos caros en falsos positivos baratos. Declarar maligno un caso es más barato en valor esperado siempre que \(P(\text{malignant}) > c_{FP}/(c_{FP} + c_{FN})\); para 10 : 1, \(1/11 \approx 0.09\) (el mínimo empírico en el conjunto de prueba fue de alrededor de 0.05).</details>

<details><summary>11. k-means concordó con la biopsia en el 93 % de los aspirados sin usar etiquetas. Un colega concluye que las etiquetas eran innecesarias. ¿Qué está mal?</summary>La concordancia solo pudo medirse <em>porque</em> existían etiquetas; sin ellas, nada indica que los grupos signifiquen diagnóstico en lugar de tamaño. Además, a k-means se le indicó \(k = 2\), y los métodos supervisados fueron al menos igual de buenos y, además, entregan una probabilidad.</details>

<details><summary>12. En el pasillo con \(\gamma = 0.9\), calcula \(V(1)\) si la salida derecha pagara 0.5 en lugar de 1.0, y la política en la posición 1.</summary>\(V(3) = 0.5\), \(V(2) = 0.45\), \(V(1) = \max(0.3, 0.405) = 0.405\): sigue siendo derecha, por poco; con \(\gamma = 0.75\), \(0.28 < 0.3\) y gira a la izquierda.</details>

## Glosario

<dl class="gloss">
<dt>Clasificador / frontera de decisión (classifier / decision boundary)</dt><dd>Una función que va de vectores de características a etiquetas de clase / la superficie donde cambia la clase predicha.</dd>
<dt>Coeficiente de determinación (coefficient of determination, R²)</dt><dd>\(1 - \text{RSS}/\text{TSS}\): la fracción de la varianza de la variable objetivo explicada por el modelo.</dd>
<dt>Validación cruzada de k particiones (k-fold cross-validation)</dt><dd>Ajustar \(k\) veces, reservando cada vez una de las \(k\) particiones, y promediar los puntajes.</dd>
<dt>Maldición de la dimensionalidad (curse of dimensionality)</dt><dd>Degradación del desempeño al agregar características con un tamaño muestral fijo, porque los datos se vuelven dispersos.</dd>
<dt>Inferencia deductiva / inductiva (deductive / inductive inference)</dt><dd>De reglas a casos (cierta, rígida) / de casos a una regla (probable, generalizadora); el aprendizaje es inductivo.</dd>
<dt>Factor de descuento (discount factor, γ)</dt><dd>El factor por el cual se devalúa una recompensa un paso en el futuro respecto de una inmediata.</dd>
<dt>Descenso de gradiente (gradient descent)</dt><dd>\(\theta \leftarrow \theta - \eta\nabla L(\theta)\); el descenso de gradiente estocástico estima el gradiente a partir de un minilote.</dd>
<dt>IC50 / pIC50</dt><dd>Concentración que inhibe a la mitad la actividad de una diana / su logaritmo decimal negativo en mol/L, la variable objetivo habitual en QSAR.</dd>
<dt>Tasa de aprendizaje (learning rate, η)</dt><dd>El tamaño del paso del descenso de gradiente; si es demasiado pequeña es lento, si es demasiado grande diverge.</dd>
<dt>Mínimos cuadrados / ecuaciones normales (least squares / normal equations)</dt><dd>Minimizar la suma de los residuos al cuadrado / el sistema lineal \(\mathbf{X}^\top\mathbf{X}\boldsymbol\beta = \mathbf{X}^\top\mathbf{y}\) que lo resuelve.</dd>
<dt>Pérdida logarítmica (log-loss, cross-entropy)</dt><dd>La media negativa de la log-probabilidad asignada a las etiquetas observadas; la pérdida de la regresión logística y de los clasificadores neuronales.</dd>
<dt>Regresión logística (logistic regression)</dt><dd>\(P(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{x}^\top\boldsymbol\beta)\) ajustada por máxima verosimilitud; los coeficientes son logaritmos de razones de odds.</dd>
<dt>Sobreajuste / subajuste (overfitting / underfitting)</dt><dd>Ajustar el ruido de la muestra (error de entrenamiento bajo, error de prueba alto) / ser demasiado rígido para el patrón (error alto en ambos).</dd>
<dt>Política / función de valor (policy / value function)</dt><dd>Una correspondencia de estados a acciones / la recompensa acumulada descontada esperada desde un estado.</dd>
<dt>Regresión / clasificación (regression / classification)</dt><dd>Aprendizaje supervisado con una variable objetivo continua / con una variable objetivo categórica (clase, etiqueta).</dd>
<dt>Regularización (regularisation, ridge)</dt><dd>Una penalización sobre el tamaño de los parámetros que se suma a la pérdida; ridge usa \(\lambda\lVert\boldsymbol\beta\rVert^2\), es decir, \(\lambda\mathbf{I}\) en las ecuaciones normales.</dd>
<dt>Sigmoide / softmax (sigmoid / softmax)</dt><dd>\(\sigma(z) = 1/(1 + e^{-z})\), que transforma un puntaje en una probabilidad (inversa: logit) / su generalización a \(K\) clases.</dd>
<dt>Datos estructurados / no estructurados (structured / unstructured data)</dt><dd>Datos con un modelo predefinido (tablas) / sin él (texto, imágenes, señales), convertidos mediante extracción de características.</dd>
<dt>Aprendizaje supervisado / no supervisado / por refuerzo (supervised / unsupervised / reinforcement learning)</dt><dd>Aprender a partir de pares etiquetados / solo de entradas / de recompensas obtenidas al actuar.</dd>
<dt>Conjunto de entrenamiento / validación / prueba (training / validation / test set)</dt><dd>Datos usados para ajustar los parámetros / para elegir los ajustes de flexibilidad / para informar una sola vez el desempeño final.</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">Mitchell TM. <em>Machine Learning</em>. New York: McGraw-Hill; 1997. Capítulo 1. <a href="http://www.cs.cmu.edu/~tom/mlbook.html">www.cs.cmu.edu/~tom/mlbook.html</a> (libro; sin DOI — página de la editorial/del autor)</li>
<li id="r2">Samuel AL. Some studies in machine learning using the game of checkers. <em>IBM J Res Dev</em>. 1959;3(3):210–229. <a href="https://doi.org/10.1147/rd.33.0210">doi:10.1147/rd.33.0210</a></li>
<li id="r3">Domingos P. A few useful things to know about machine learning. <em>Commun ACM</em>. 2012;55(10):78–87. <a href="https://doi.org/10.1145/2347736.2347755">doi:10.1145/2347736.2347755</a></li>
<li id="r4">Donoho D. 50 years of data science. <em>J Comput Graph Stat</em>. 2017;26(4):745–766. <a href="https://doi.org/10.1080/10618600.2017.1384734">doi:10.1080/10618600.2017.1384734</a></li>
<li id="r5">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r6">Lipinski CA, Lombardo F, Dominy BW, Feeney PJ. Experimental and computational approaches to estimate solubility and permeability in drug discovery and development settings. <em>Adv Drug Deliv Rev</em>. 2001;46(1–3):3–26. <a href="https://doi.org/10.1016/S0169-409X(00)00129-0">doi:10.1016/S0169-409X(00)00129-0</a></li>
<li id="r7">Wishart DS, Feunang YD, Guo AC, et al. DrugBank 5.0: a major update to the DrugBank database for 2018. <em>Nucleic Acids Res</em>. 2018;46(D1):D1074–D1082. <a href="https://doi.org/10.1093/nar/gkx1037">doi:10.1093/nar/gkx1037</a></li>
<li id="r8">Mendez D, Gaulton A, Bento AP, et al. ChEMBL: towards direct deposition of bioassay data. <em>Nucleic Acids Res</em>. 2019;47(D1):D930–D940. <a href="https://doi.org/10.1093/nar/gky1075">doi:10.1093/nar/gky1075</a></li>
<li id="r9">Hughes GF. On the mean accuracy of statistical pattern recognizers. <em>IEEE Trans Inf Theory</em>. 1968;14(1):55–63. <a href="https://doi.org/10.1109/TIT.1968.1054102">doi:10.1109/TIT.1968.1054102</a></li>
<li id="r10">Agrawal R, Imieliński T, Swami A. Mining association rules between sets of items in large databases. <em>ACM SIGMOD Rec</em>. 1993;22(2):207–216. <a href="https://doi.org/10.1145/170036.170072">doi:10.1145/170036.170072</a></li>
<li id="r11">Sutton RS, Barto AG. <em>Reinforcement Learning: An Introduction</em>. 2nd ed. Cambridge, MA: MIT Press; 2018. <a href="http://incompleteideas.net/book/the-book-2nd.html">incompleteideas.net/book/the-book-2nd.html</a> (libro; sin DOI — página de los autores con el texto completo)</li>
<li id="r12">Komorowski M, Celi LA, Badawi O, Gordon AC, Faisal AA. The Artificial Intelligence Clinician learns optimal treatment strategies for sepsis in intensive care. <em>Nat Med</em>. 2018;24(11):1716–1720. <a href="https://doi.org/10.1038/s41591-018-0213-5">doi:10.1038/s41591-018-0213-5</a></li>
<li id="r13">Chapelle O, Schölkopf B, Zien A, eds. <em>Semi-Supervised Learning</em>. Cambridge, MA: MIT Press; 2006. <a href="https://doi.org/10.7551/mitpress/9780262033589.001.0001">doi:10.7551/mitpress/9780262033589.001.0001</a></li>
<li id="r14">Krishnan R, Rajpurkar P, Topol EJ. Self-supervised learning in medicine and healthcare. <em>Nat Biomed Eng</em>. 2022;6(12):1346–1352. <a href="https://doi.org/10.1038/s41551-022-00914-1">doi:10.1038/s41551-022-00914-1</a></li>
<li id="r15">Jain AK, Duin RPW, Mao J. Statistical pattern recognition: a review. <em>IEEE Trans Pattern Anal Mach Intell</em>. 2000;22(1):4–37. <a href="https://doi.org/10.1109/34.824819">doi:10.1109/34.824819</a></li>
<li id="r16">Rosenblatt F. The perceptron: a probabilistic model for information storage and organization in the brain. <em>Psychol Rev</em>. 1958;65(6):386–408. <a href="https://doi.org/10.1037/h0042519">doi:10.1037/h0042519</a></li>
<li id="r17">Holland JH. Genetic algorithms. <em>Sci Am</em>. 1992;267(1):66–72. <a href="https://doi.org/10.1038/scientificamerican0792-66">doi:10.1038/scientificamerican0792-66</a></li>
<li id="r18">Hand DJ. Classifier technology and the illusion of progress. <em>Stat Sci</em>. 2006;21(1):1–14. <a href="https://doi.org/10.1214/088342306000000060">doi:10.1214/088342306000000060</a></li>
<li id="r19">MacQueen J. Some methods for classification and analysis of multivariate observations. In: <em>Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability</em>, vol. 1. Berkeley: University of California Press; 1967:281–297. <a href="https://projecteuclid.org/euclid.bsmsp/1200512992">projecteuclid.org/euclid.bsmsp/1200512992</a> (sin DOI — página de Project Euclid)</li>
<li id="r20">Lloyd SP. Least squares quantization in PCM. <em>IEEE Trans Inf Theory</em>. 1982;28(2):129–137. <a href="https://doi.org/10.1109/TIT.1982.1056489">doi:10.1109/TIT.1982.1056489</a></li>
<li id="r21">Efron B, Hastie T, Johnstone I, Tibshirani R. Least angle regression. <em>Ann Stat</em>. 2004;32(2):407–499. <a href="https://doi.org/10.1214/009053604000000067">doi:10.1214/009053604000000067</a></li>
<li id="r22">Galton F. Regression towards mediocrity in hereditary stature. <em>J Anthropol Inst G B Irel</em>. 1886;15:246–263. <a href="https://doi.org/10.2307/2841583">doi:10.2307/2841583</a></li>
<li id="r23">Stigler SM. Gauss and the invention of least squares. <em>Ann Stat</em>. 1981;9(3):465–474. <a href="https://doi.org/10.1214/aos/1176345451">doi:10.1214/aos/1176345451</a></li>
<li id="r24">Hoerl AE, Kennard RW. Ridge regression: biased estimation for nonorthogonal problems. <em>Technometrics</em>. 1970;12(1):55–67. <a href="https://doi.org/10.1080/00401706.1970.10488634">doi:10.1080/00401706.1970.10488634</a></li>
<li id="r25">Robbins H, Monro S. A stochastic approximation method. <em>Ann Math Stat</em>. 1951;22(3):400–407. <a href="https://doi.org/10.1214/aoms/1177729586">doi:10.1214/aoms/1177729586</a></li>
<li id="r26">Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning: Data Mining, Inference, and Prediction</em>. 2nd ed. New York: Springer; 2009. <a href="https://doi.org/10.1007/978-0-387-84858-7">doi:10.1007/978-0-387-84858-7</a></li>
<li id="r27">Cox DR. The regression analysis of binary sequences. <em>J R Stat Soc Ser B</em>. 1958;20(2):215–232. <a href="https://doi.org/10.1111/j.2517-6161.1958.tb00292.x">doi:10.1111/j.2517-6161.1958.tb00292.x</a></li>
<li id="r28">Vickers AJ, Elkin EB. Decision curve analysis: a novel method for evaluating prediction models. <em>Med Decis Making</em>. 2006;26(6):565–574. <a href="https://doi.org/10.1177/0272989X06295361">doi:10.1177/0272989X06295361</a></li>
<li id="r29">Van Calster B, McLernon DJ, van Smeden M, Wynants L, Steyerberg EW. Calibration: the Achilles heel of predictive analytics. <em>BMC Med</em>. 2019;17:230. <a href="https://doi.org/10.1186/s12916-019-1466-7">doi:10.1186/s12916-019-1466-7</a></li>
<li id="r30">Wilson PWF, D'Agostino RB, Levy D, Belanger AM, Silbershatz H, Kannel WB. Prediction of coronary heart disease using risk factor categories. <em>Circulation</em>. 1998;97(18):1837–1847. <a href="https://doi.org/10.1161/01.CIR.97.18.1837">doi:10.1161/01.CIR.97.18.1837</a></li>
<li id="r31">Le Gall J-R, Lemeshow S, Saulnier F. A new Simplified Acute Physiology Score (SAPS II) based on a European/North American multicenter study. <em>JAMA</em>. 1993;270(24):2957–2963. <a href="https://doi.org/10.1001/jama.1993.03510240069035">doi:10.1001/jama.1993.03510240069035</a></li>
<li id="r32">Gail MH, Brinton LA, Byar DP, et al. Projecting individualized probabilities of developing breast cancer for white females who are being examined annually. <em>J Natl Cancer Inst</em>. 1989;81(24):1879–1886. <a href="https://doi.org/10.1093/jnci/81.24.1879">doi:10.1093/jnci/81.24.1879</a></li>
<li id="r33">Collins GS, Reitsma JB, Altman DG, Moons KGM. Transparent Reporting of a multivariable prediction model for Individual Prognosis Or Diagnosis (TRIPOD): the TRIPOD statement. <em>Ann Intern Med</em>. 2015;162(1):55–63. <a href="https://doi.org/10.7326/M14-0697">doi:10.7326/M14-0697</a></li>
<li id="r34">Alpaydin E, Kaynak C. Optical Recognition of Handwritten Digits [dataset]. UCI Machine Learning Repository; 1998. <a href="https://doi.org/10.24432/C50P49">doi:10.24432/C50P49</a></li>
<li id="r35">Watkins CJCH, Dayan P. Q-learning. <em>Mach Learn</em>. 1992;8(3–4):279–292. <a href="https://doi.org/10.1007/BF00992698">doi:10.1007/BF00992698</a></li>
<li id="r36">R Core Team. R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; sin DOI)</li>
<li id="r37">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. Paquete de R. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r38">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. New York: Springer; 2002 (los paquetes <code>MASS</code>, <code>class</code> y <code>nnet</code>). <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r39">James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. New York: Springer; 2021. <a href="https://doi.org/10.1007/978-1-0716-1418-1">doi:10.1007/978-1-0716-1418-1</a></li>
<li id="r40">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
</ol>

<h4>Libros de texto</h4>
<ul>
<li>Mitchell TM. <em>Machine Learning</em>. McGraw-Hill; 1997. — Capítulo 1 para la definición de aprendizaje; capítulos 4 y 13 para el descenso de gradiente y el aprendizaje por refuerzo.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — Los capítulos 2–4 cubren las Secciones 6–9 con código R; PDF gratuito en statlearning.com.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Capítulos 3–4 para la regresión lineal y logística y ridge; PDF gratuito.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS). 4th ed. Springer; 2002. — La referencia para las funciones de modelado de R usadas aquí (<code>lda</code>, <code>nnet</code>, <code>multinom</code>).</li>
<li>Sutton RS, Barto AG. <em>Reinforcement Learning: An Introduction</em>. 2nd ed. MIT Press; 2018. — Capítulos 3–4 y 6 para la iteración de valores y el Q-learning; gratuito en línea.</li>
<li>Bishop CM. <em>Pattern Recognition and Machine Learning</em>. Springer; 2006. — Capítulo 1 para el experimento polinomial de la Sección 8 en su forma original.</li>
</ul>

<h4>Recursos en línea</h4>
<ul>
<li>Vista de tareas (task view) de CRAN "Machine Learning &amp; Statistical Learning" — <a href="https://cran.r-project.org/view=MachineLearning">cran.r-project.org/view=MachineLearning</a>.</li>
<li>ChEMBL — <a href="https://www.ebi.ac.uk/chembl/">ebi.ac.uk/chembl</a>; BindingDB — <a href="https://www.bindingdb.org">bindingdb.org</a>; DrugBank — <a href="https://go.drugbank.com">go.drugbank.com</a>; Drug Repurposing Hub — <a href="https://clue.io/repurposing">clue.io/repurposing</a>.</li>
<li>UCI Machine Learning Repository — <a href="https://archive.ics.uci.edu">archive.ics.uci.edu</a> (los datos originales de cáncer de mama de Wisconsin y de Boston housing).</li>
<li>Funciones de riesgo del Framingham Heart Study — <a href="https://www.framinghamheartstudy.org/fhs-risk-functions/">framinghamheartstudy.org/fhs-risk-functions</a>.</li>
</ul>
